# ARC-AGI-2 Program No. 077
## September Hidden-Rerun Recovery — Normal CUDA Program024 Primary Only

Program077 preserves the exact Program024 Primary solver, normal non-deterministic CUDA backend, and exact two-guess KGMon selection from the previous anchor. It fixes the hidden competition rerun failure mode: a lower hidden candidate-presence ratio, partial Primary completion, or another recoverable quality-audit warning is recorded but no longer converted into an unhandled exception before `submission.json` is written.

The normal path salvages every valid atomic Primary output and lets `ArcDataset.get_submission` fill missing attempts with the schema-valid `[[0]]` placeholder. If the isolated finalizer itself fails or exceeds fifteen minutes, a last-resort emergency writer still produces a complete task/test/attempt structure from the hidden challenge file. Schema or challenge-file failures remain hard errors because no valid competition submission can be constructed in those cases.

The notebook safety cap remains 9 hours 30 minutes. Primary work stops at 9 hours 15 minutes, preserving at least fifteen minutes for scanning, checkpointing, recovery, validation, and writing. Deep and post-hoc ranking remain disabled. Submission is never automatic.


In [ ]:
from __future__ import annotations

import base64
import csv
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

PROGRAM_NO = 77
BASE_SEED = 42
PYTHON_HASH_SEED = "0"
SEED_CONTRACT = {
    "base_seed": BASE_SEED,
    "python_hash_seed": PYTHON_HASH_SEED,
    "worker_init_reset": True,
    "task_dequeue_reset": True,
    "deterministic_algorithms_enabled": False,
    "backend_policy_changed": False,
}
NOTEBOOK_START_TIME = time.time()
HARD_RUNTIME_LIMIT_SECONDS = 9 * 3600 + 30 * 60
FINALIZATION_RESERVE_SECONDS = 15 * 60
hard_end_time = NOTEBOOK_START_TIME + HARD_RUNTIME_LIMIT_SECONDS
global_end_time = hard_end_time - FINALIZATION_RESERVE_SECONDS
MODEL_PATH = Path("/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1")
RUNTIME_HINTS = [
    Path("/kaggle/usr/lib/notebooks/mirzamilanfarabi/pip_install_unsloth_flash_patch"),
    Path("/kaggle/usr/lib/notebooks/sorokin/pip_install_unsloth_flash_patch"),
]
PROFILE_ROOT = Path("/kaggle/working/program077_runtime_profiles")
WORK_ROOT = Path("/kaggle/working")
STALE_SUBMISSION_PATH = Path("/kaggle/working/submission.json")
STALE_SUBMISSION_PATH.unlink(missing_ok=True)
print("PROGRAM077_STALE_SUBMISSION_CLEARED=" + str(STALE_SUBMISSION_PATH))


def early_l4x4_gate() -> dict:
    try:
        probe = subprocess.run(
            ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
            text=True,
            capture_output=True,
            timeout=10,
            check=False,
        )
    except Exception as exc:
        raise RuntimeError(
            f"PROGRAM077_EARLY_GPU_GATE_FAILED=nvidia-smi probe error: {type(exc).__name__}: {exc}"
        ) from exc
    names = [line.strip() for line in probe.stdout.splitlines() if line.strip()]
    expected = ["NVIDIA L4"] * 4
    report = {
        "command_returncode": probe.returncode,
        "device_count": len(names),
        "device_names": names,
        "expected": expected,
    }
    if probe.returncode != 0 or names != expected:
        stderr_tail = " | ".join(probe.stderr.strip().splitlines()[-5:])
        report["stderr_tail"] = stderr_tail
        raise RuntimeError("PROGRAM077_EARLY_GPU_GATE_FAILED=" + json.dumps(report, sort_keys=True))
    print("PROGRAM077_EARLY_GPU_GATE_OK=" + json.dumps(report, sort_keys=True))
    return report


EARLY_GPU_REPORT = early_l4x4_gate()

NATIVE_BINARY_ANCHORS = {
    "torch", "triton", "torchvision", "torchaudio", "tokenizers", "safetensors",
    "numpy", "scipy", "pandas", "pyarrow", "sentencepiece", "google", "PIL",
}
GENERIC_SHARED_LIBRARY_ALLOWLIST = {"bitsandbytes"}


# Kaggle can expose either of two known, internally coherent Python package
# families under the mounted Unsloth runtime. Program 073 searches every
# mounted source root before GPU work. It prefers the stable program047_legacy
# family used by the successful Program 071 run, with the current family kept
# only as a fully verified fallback. Every distribution is verified from RECORD.
RUNTIME_FAMILIES = [
    (
        "program047_legacy",
        {
            "accelerate": {
                "version": "1.11.0",
                "official_wheel_filename": "accelerate-1.11.0-py3-none-any.whl",
                "official_wheel_sha256": "a628fa6beb069b8e549460fc449135d5bd8d73e7a11fd09f0bc9fc4ace7f06f1",
            },
            "peft": {
                "version": "0.17.1",
                "official_wheel_filename": "peft-0.17.1-py3-none-any.whl",
                "official_wheel_sha256": "3d129d64def3d74779c32a080d2567e5f7b674e77d546e3585138216d903f99e",
            },
            "bitsandbytes": {
                "version": "0.48.2",
                "official_wheel_filename": "bitsandbytes-0.48.2-py3-none-manylinux_2_24_x86_64.whl",
                "official_wheel_sha256": "cd289562cb7308ee2a707e6884fecca9bbbcfc9ec33a86df2a45e0779692c1a3",
            },
        },
    ),
    (
        "program050_current",
        {
            "accelerate": {
                "version": "1.13.0",
                "official_wheel_filename": "accelerate-1.13.0-py3-none-any.whl",
                "official_wheel_sha256": "cf1a3efb96c18f7b152eb0fa7490f3710b19c3f395699358f08decca2b8b62e0",
            },
            "peft": {
                "version": "0.18.1",
                "official_wheel_filename": "peft-0.18.1-py3-none-any.whl",
                "official_wheel_sha256": "0bf06847a3551e3019fc58c440cffc9a6b73e6e2962c95b52e224f77bbdb50f1",
            },
            "bitsandbytes": {
                "version": "0.49.2",
                "official_wheel_filename": "bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl",
                "official_wheel_sha256": "54b771f06e1a3c73af5c7f16ccf0fc23a846052813d4b008d10cb6e017dd1c8c",
            },
        },
    ),
]
PINNED_RUNTIME_DISTRIBUTIONS: dict[str, dict[str, str]] = {}



def env_flag(name: str) -> bool:
    value = os.getenv(name)
    return value is not None and value.strip().lower() in {"1", "true", "yes", "y", "on"}


def canonical_name(value: str) -> str:
    return re.sub(r"[-_.]+", "-", value).lower()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_named_file(names: list[str]) -> Path:
    roots = [
        Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2"),
        Path("/kaggle/input/arc-prize-2026-arc-agi-2"),
    ]
    for root in roots:
        for name in names:
            candidate = root / name
            if candidate.is_file():
                return candidate.resolve()
    input_root = Path("/kaggle/input")
    for name in names:
        matches = sorted(input_root.rglob(name))
        if matches:
            return matches[0].resolve()
    raise FileNotFoundError(f"Could not resolve any of {names} below /kaggle/input")


def resolve_ptxas() -> Path:
    candidates = [
        Path("/usr/local/cuda/bin/ptxas"),
        Path("/usr/local/cuda-12.8/bin/ptxas"),
        Path("/usr/local/cuda-12.5/bin/ptxas"),
    ]
    discovered = shutil.which("ptxas")
    if discovered:
        candidates.append(Path(discovered))
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except OSError:
            continue
        if resolved in seen:
            continue
        seen.add(resolved)
        if resolved.is_file() and os.access(resolved, os.X_OK):
            return resolved
    raise FileNotFoundError("ptxas was not found; aborting before GPU model loading")


def discover_legacy_roots() -> list[Path]:
    candidates = list(RUNTIME_HINTS)
    notebook_root = Path("/kaggle/usr/lib/notebooks")
    if notebook_root.is_dir():
        candidates.extend(sorted(notebook_root.glob("**/pip_install_unsloth_flash_patch")))
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        for unsloth_init in sorted(input_root.glob("**/unsloth/__init__.py")):
            candidates.append(unsloth_init.parent.parent)
    roots: list[Path] = []
    seen: set[Path] = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except OSError:
            continue
        if resolved in seen:
            continue
        seen.add(resolved)
        if (resolved / "unsloth" / "__init__.py").is_file():
            roots.append(resolved)
    if not roots:
        raise FileNotFoundError(
            "PROGRAM077_LEGACY_RUNTIME_MISSING: no mounted Unsloth runtime root was found"
        )
    return roots


def read_metadata_name(dist_info: Path) -> str | None:
    metadata = dist_info / "METADATA"
    if not metadata.is_file():
        return None
    for line in metadata.read_text(encoding="utf-8", errors="replace").splitlines():
        if line.startswith("Name:"):
            return line.split(":", 1)[1].strip()
    return None



def read_metadata_version(dist_info: Path) -> str | None:
    metadata = dist_info / "METADATA"
    if not metadata.is_file():
        return None
    for line in metadata.read_text(encoding="utf-8", errors="replace").splitlines():
        if line.startswith("Version:"):
            return line.split(":", 1)[1].strip()
    return None


def distribution_index(
    source_root: Path,
    runtime_lock: dict[str, dict[str, str]],
    runtime_family: str,
) -> tuple[dict[str, Path], dict[str, str], dict[str, list[str]]]:
    catalog: dict[str, list[tuple[str, Path]]] = {}
    module_to_dist: dict[str, str] = {}
    for dist_info in sorted(source_root.glob("*.dist-info")):
        name = read_metadata_name(dist_info)
        version = read_metadata_version(dist_info)
        if not name or not version:
            continue
        canonical = canonical_name(name)
        catalog.setdefault(canonical, []).append((version, dist_info))
        top_level = dist_info / "top_level.txt"
        modules: list[str] = []
        if top_level.is_file():
            modules = [
                line.strip()
                for line in top_level.read_text(encoding="utf-8", errors="replace").splitlines()
                if line.strip()
            ]
        if not modules:
            record = dist_info / "RECORD"
            if record.is_file():
                with record.open("r", encoding="utf-8", errors="replace", newline="") as handle:
                    for row in csv.reader(handle):
                        if not row:
                            continue
                        parts = Path(row[0]).parts
                        first = parts[0] if parts else ""
                        if first and not first.endswith((".dist-info", ".data")):
                            module_name = first[:-3] if first.endswith(".py") else first
                            if module_name.isidentifier():
                                modules.append(module_name)
        for module_name in modules:
            module_to_dist.setdefault(module_name.split(".", 1)[0], canonical)

    available_versions = {
        canonical: sorted({version for version, _ in entries})
        for canonical, entries in catalog.items()
    }
    pinned_matches: dict[str, Path] = {}
    unavailable: dict[str, dict] = {}
    for canonical, lock in sorted(runtime_lock.items()):
        expected = lock["version"]
        entries = catalog.get(canonical, [])
        matches = [dist_info for version, dist_info in entries if version == expected]
        if len(matches) != 1:
            unavailable[canonical] = {
                "expected_version": expected,
                "available_versions": available_versions.get(canonical, []),
                "matching_dist_info_count": len(matches),
            }
        else:
            pinned_matches[canonical] = matches[0]
    if unavailable:
        raise RuntimeError(
            "PROGRAM077_RUNTIME_FAMILY_UNAVAILABLE="
            + json.dumps({
                "runtime_family": runtime_family,
                "source_root": str(source_root),
                "unavailable": unavailable,
                "expected_versions": {
                    name: lock["version"] for name, lock in sorted(runtime_lock.items())
                },
            }, sort_keys=True)
        )

    by_name: dict[str, Path] = {}
    for canonical, entries in catalog.items():
        if canonical in pinned_matches:
            by_name[canonical] = pinned_matches[canonical]
        else:
            by_name[canonical] = sorted(entries, key=lambda item: item[1].name)[-1][1]
    return by_name, module_to_dist, available_versions

def distribution_roots(source_root: Path, dist_info: Path) -> list[Path]:
    roots: set[Path] = set()
    top_level = dist_info / "top_level.txt"
    if top_level.is_file():
        for line in top_level.read_text(encoding="utf-8", errors="replace").splitlines():
            module = line.strip()
            if not module:
                continue
            for candidate in (source_root / module, source_root / f"{module}.py"):
                if candidate.exists():
                    roots.add(candidate)
    record = dist_info / "RECORD"
    if record.is_file():
        with record.open("r", encoding="utf-8", errors="replace", newline="") as handle:
            for row in csv.reader(handle):
                if not row:
                    continue
                rel = Path(row[0])
                if rel.is_absolute() or ".." in rel.parts or not rel.parts:
                    continue
                first = rel.parts[0]
                if first.endswith((".dist-info", ".data")):
                    continue
                candidate = source_root / first
                if candidate.exists():
                    roots.add(candidate)
    roots.add(dist_info)
    return sorted(roots, key=lambda path: path.name)



def _decode_record_sha256(value: str) -> bytes:
    padding = "=" * (-len(value) % 4)
    return base64.urlsafe_b64decode(value + padding)


def verify_distribution_record(
    source_root: Path,
    dist_name: str,
    by_name: dict[str, Path],
) -> dict:
    canonical = canonical_name(dist_name)
    lock = PINNED_RUNTIME_DISTRIBUTIONS[canonical]
    dist_info = by_name.get(canonical)
    if dist_info is None:
        raise RuntimeError(f"PROGRAM077_PINNED_DIST_INFO_MISSING={canonical}")
    actual_version = read_metadata_version(dist_info)
    if actual_version != lock["version"]:
        raise RuntimeError(
            "PROGRAM077_PINNED_VERSION_MISMATCH="
            + json.dumps({
                "distribution": canonical,
                "expected": lock["version"],
                "actual": actual_version,
                "dist_info": str(dist_info),
            }, sort_keys=True)
        )

    record = dist_info / "RECORD"
    metadata = dist_info / "METADATA"
    if not record.is_file() or not metadata.is_file():
        raise RuntimeError(f"PROGRAM077_PINNED_METADATA_INCOMPLETE={dist_info}")

    verified_entries: list[tuple[str, str, int]] = []
    skipped_external = 0
    unhashed_entries = 0
    with record.open("r", encoding="utf-8", errors="strict", newline="") as handle:
        for row in csv.reader(handle):
            if not row:
                continue
            rel_text = row[0]
            hash_spec = row[1] if len(row) > 1 else ""
            size_text = row[2] if len(row) > 2 else ""
            rel = Path(rel_text)
            if rel.is_absolute() or ".." in rel.parts or not rel.parts:
                skipped_external += 1
                continue
            target = source_root / rel
            if not hash_spec:
                unhashed_entries += 1
                continue
            if "=" not in hash_spec:
                raise RuntimeError(f"PROGRAM077_RECORD_HASH_FORMAT_INVALID={canonical}:{rel_text}")
            algorithm, encoded = hash_spec.split("=", 1)
            if algorithm != "sha256":
                raise RuntimeError(f"PROGRAM077_RECORD_HASH_ALGORITHM_INVALID={canonical}:{algorithm}")
            if not target.is_file():
                raise RuntimeError(f"PROGRAM077_RECORD_FILE_MISSING={canonical}:{target}")
            actual_digest = hashlib.sha256(target.read_bytes()).digest()
            expected_digest = _decode_record_sha256(encoded)
            if actual_digest != expected_digest:
                raise RuntimeError(
                    "PROGRAM077_RECORD_FILE_HASH_MISMATCH="
                    + json.dumps({
                        "distribution": canonical,
                        "file": rel.as_posix(),
                        "expected": expected_digest.hex(),
                        "actual": actual_digest.hex(),
                    }, sort_keys=True)
                )
            actual_size = target.stat().st_size
            if size_text and actual_size != int(size_text):
                raise RuntimeError(
                    "PROGRAM077_RECORD_FILE_SIZE_MISMATCH="
                    + json.dumps({
                        "distribution": canonical,
                        "file": rel.as_posix(),
                        "expected": int(size_text),
                        "actual": actual_size,
                    }, sort_keys=True)
                )
            verified_entries.append((rel.as_posix(), actual_digest.hex(), actual_size))

    if not verified_entries:
        raise RuntimeError(f"PROGRAM077_RECORD_NO_VERIFIED_FILES={canonical}")
    tree_digest = hashlib.sha256()
    for rel_text, digest_hex, size in sorted(verified_entries):
        tree_digest.update(f"{rel_text}\0{digest_hex}\0{size}\n".encode("utf-8"))
    return {
        "distribution": canonical,
        "version": actual_version,
        "dist_info": dist_info.name,
        "metadata_sha256": sha256_file(metadata),
        "record_sha256": sha256_file(record),
        "verified_file_count": len(verified_entries),
        "verified_byte_count": sum(size for _, _, size in verified_entries),
        "record_tree_sha256": tree_digest.hexdigest(),
        "unhashed_record_entries": unhashed_entries,
        "skipped_external_record_entries": skipped_external,
        "official_wheel_filename_reference": lock["official_wheel_filename"],
        "official_wheel_sha256_reference": lock["official_wheel_sha256"],
    }


def verify_pinned_runtime(source_root: Path, by_name: dict[str, Path]) -> dict[str, dict]:
    return {
        canonical: verify_distribution_record(source_root, canonical, by_name)
        for canonical in sorted(PINNED_RUNTIME_DISTRIBUTIONS)
    }

def incompatible_extension(root: Path, dist_name: str) -> str | None:
    files = [root] if root.is_file() else list(root.rglob("*"))
    for path in files:
        if not path.is_file() or path.suffix != ".so":
            continue
        lower = path.name.lower()
        if re.search(r"cpython-3(?:9|10|11)-", lower):
            return str(path)
        if dist_name not in GENERIC_SHARED_LIBRARY_ALLOWLIST and "cpython-" in lower:
            return str(path)
    return None


def copy_distribution(
    source_root: Path,
    overlay_root: Path,
    dist_name: str,
    by_name: dict[str, Path],
) -> list[str]:
    canonical = canonical_name(dist_name)
    if canonical in NATIVE_BINARY_ANCHORS:
        raise RuntimeError(f"Refusing to overlay native binary anchor: {dist_name}")
    dist_info = by_name.get(canonical)
    if dist_info is None:
        raise FileNotFoundError(f"Legacy distribution not found: {dist_name}")
    copied: list[str] = []
    for source in distribution_roots(source_root, dist_info):
        bad = incompatible_extension(source, canonical)
        if bad:
            raise RuntimeError(f"Legacy distribution {dist_name} contains a Python-ABI extension: {bad}")
        destination = overlay_root / source.name
        if destination.exists():
            continue
        if source.is_dir():
            shutil.copytree(source, destination, symlinks=False)
        else:
            shutil.copy2(source, destination)
        copied.append(source.name)
    return copied


def sanitize_pythonpath(current: str, source_root: Path, overlay_root: Path) -> str:
    result = [str(overlay_root)]
    seen = {overlay_root.resolve()}
    for raw in current.split(os.pathsep):
        if not raw:
            continue
        try:
            resolved = Path(raw).resolve()
        except OSError:
            continue
        if resolved == source_root or source_root in resolved.parents or resolved in seen:
            continue
        seen.add(resolved)
        result.append(raw)
    return os.pathsep.join(result)


def patch_legacy_transformers(overlay_root: Path) -> list[str]:
    patches: list[str] = []
    table = overlay_root / "transformers" / "dependency_versions_table.py"
    if table.is_file():
        text = table.read_text(encoding="utf-8")
        updated = re.sub(
            r'(tokenizers>=0\.\d+(?:\.\d+)?,<)0\.22',
            r'\g<1>0.23',
            text,
        )
        if updated != text:
            table.write_text(updated, encoding="utf-8")
            patches.append("transformers:tokenizers<0.23")
    return patches



def install_qwen3_official_kv_cache_patch(overlay_root: Path) -> dict:
    '''Patch only the broken legacy Qwen3 cached-attention call site.

    The copied Unsloth 2025.9.7 source has one cached-inference call to
    ``flash_attn_func(Qnn, Knn, Vnn)`` even when FlashAttention is unavailable.
    Program 073 replaces that single call with an in-module helper matching the
    current Unsloth Qwen3 branch structure, while leaving the surrounding legacy
    RoPE, KV-cache storage, output projection, and Program 024 solver untouched.
    '''
    qwen3_path = overlay_root / "unsloth" / "models" / "qwen3.py"
    if not qwen3_path.is_file():
        raise FileNotFoundError(f"Program 073 Qwen3 source is missing: {qwen3_path}")

    marker = "# PROGRAM077_QWEN3_OFFICIAL_KV_CACHE_V1"
    expected_original_sha256 = "434edd417020c58f15cfa94c10a0e5abd63e95a9ec83276988b876e63942ae09"
    original = qwen3_path.read_text(encoding="utf-8")
    before_hash = hashlib.sha256(original.encode("utf-8")).hexdigest()
    installed = False
    replacements = 0

    if marker in original:
        patched = original
    else:
        if before_hash != expected_original_sha256:
            raise RuntimeError(
                "PROGRAM077_QWEN3_SOURCE_HASH_MISMATCH="
                + json.dumps({
                    "expected": expected_original_sha256,
                    "actual": before_hash,
                    "path": str(qwen3_path),
                }, sort_keys=True)
            )
        legacy_call = "flash_attn_func(Qnn, Knn, Vnn)"
        if original.count(legacy_call) != 1:
            raise RuntimeError(
                f"PROGRAM077_QWEN3_LEGACY_CALL_COUNT={original.count(legacy_call)}"
            )
        replacement = (
            "_program077_qwen3_cached_attention("
            "Qnn, Knn, Vnn, "
            "attention_mask=locals().get('attention_mask', None), "
            "sliding_window=getattr(self.config, 'sliding_window', None))"
        )
        body = original.replace(legacy_call, replacement, 1)
        replacements = 1
        helper = r'''

# PROGRAM077_QWEN3_OFFICIAL_KV_CACHE_V1
# B,S,H,D compatibility helper for the one legacy cached-inference call site.
def _program077_qwen3_cached_attention(
    q,
    k,
    v,
    *,
    attention_mask=None,
    sliding_window=None,
):
    import torch
    import torch.nn.functional as _program077_F

    if q.ndim != 4 or k.ndim != 4 or v.ndim != 4:
        raise RuntimeError(
            f"PROGRAM077_QWEN3_EXPECTED_BSHD got={q.shape},{k.shape},{v.shape}"
        )
    if q.shape[0] != k.shape[0] or k.shape[0] != v.shape[0]:
        raise RuntimeError("PROGRAM077_QWEN3_BATCH_MISMATCH")
    if k.shape[0] != v.shape[0] or k.shape[1] != v.shape[1] or k.shape[2:] != v.shape[2:]:
        raise RuntimeError("PROGRAM077_QWEN3_KV_MISMATCH")
    if q.shape[-1] != k.shape[-1]:
        raise RuntimeError("PROGRAM077_QWEN3_HEAD_DIM_MISMATCH")

    # The legacy call uses FlashAttention layout [B, S, H, D].
    q_t = q.transpose(1, 2)
    k_t = k.transpose(1, 2)
    v_t = v.transpose(1, 2)
    batch_size = q_t.shape[0]
    q_heads = q_t.shape[1]
    kv_heads = k_t.shape[1]
    if kv_heads <= 0 or q_heads % kv_heads != 0:
        raise RuntimeError(
            f"PROGRAM077_QWEN3_GQA_MISMATCH q_heads={q_heads} kv_heads={kv_heads}"
        )
    groups = q_heads // kv_heads

    # Match current Unsloth's cached Qwen3 window handling.
    if sliding_window is not None:
        try:
            window = int(sliding_window)
        except (TypeError, ValueError):
            window = 0
        if window > 0 and k_t.shape[-2] > window:
            start = k_t.shape[-2] - window
            k_t = k_t[:, :, start:, :]
            v_t = v_t[:, :, start:, :]
            if attention_mask is not None:
                attention_mask = attention_mask[..., start:]

    q_len = q_t.shape[-2]
    k_len = k_t.shape[-2]
    mask = attention_mask
    if mask is not None:
        if mask.dim() == 2:
            mask = mask[:, None, None, :].to(torch.bool)
        elif mask.dim() == 4 and mask.dtype != torch.bool:
            mask = mask.eq(0)
        elif mask.dim() not in (3, 4):
            raise RuntimeError(f"PROGRAM077_QWEN3_MASK_DIM={mask.dim()}")
        if mask.shape[-1] > k_len:
            mask = mask[..., -k_len:]
        elif mask.shape[-1] < k_len:
            raise RuntimeError(
                f"PROGRAM077_QWEN3_MASK_LENGTH mask={mask.shape[-1]} kv={k_len}"
            )

    is_causal = mask is None and q_len == k_len
    use_sdpa_gqa = groups != 1
    if (
        use_sdpa_gqa
        and isinstance(mask, torch.Tensor)
        and mask.dim() >= 3
        and mask.shape[0] > 1
    ):
        # Current Unsloth avoids SDPA-GQA drift for batched masked decode.
        use_sdpa_gqa = False

    # Current Unsloth uses explicit attention for batch one. It also expands
    # K/V whenever SDPA-GQA is unavailable or deliberately disabled.
    if batch_size == 1 or (groups != 1 and not use_sdpa_gqa):
        k_t = k_t.repeat_interleave(groups, dim=1)
        v_t = v_t.repeat_interleave(groups, dim=1)
        use_sdpa_gqa = False

    if batch_size == 1:
        scale = q_t.shape[-1] ** -0.5
        scores = torch.matmul(q_t * scale, k_t.transpose(-2, -1))
        probabilities = torch.softmax(scores, dim=-1, dtype=torch.float32).to(q_t.dtype)
        out = torch.matmul(probabilities, v_t)
    else:
        kwargs = {
            "attn_mask": mask,
            "dropout_p": 0.0,
            "is_causal": is_causal,
        }
        if use_sdpa_gqa:
            try:
                out = _program077_F.scaled_dot_product_attention(
                    q_t, k_t, v_t, enable_gqa=True, **kwargs
                )
            except TypeError:
                k_t = k_t.repeat_interleave(groups, dim=1)
                v_t = v_t.repeat_interleave(groups, dim=1)
                out = _program077_F.scaled_dot_product_attention(q_t, k_t, v_t, **kwargs)
        else:
            out = _program077_F.scaled_dot_product_attention(q_t, k_t, v_t, **kwargs)
    return out.transpose(1, 2).contiguous()

_program077_qwen3_cached_attention.__program077_backend__ = "official_style_kv_cache"
'''
        patched = body.rstrip() + "\n" + helper.lstrip("\n")
        compile(patched, str(qwen3_path), "exec")
        qwen3_path.write_text(patched, encoding="utf-8")
        installed = True

    final = qwen3_path.read_text(encoding="utf-8")
    if final.count(marker) != 1:
        raise RuntimeError(
            f"Program 073 expected one Qwen3 helper marker, found {final.count(marker)}"
        )
    if "def flash_attn_func(" in final and marker in final:
        raise RuntimeError("PROGRAM077_GENERIC_FLASH_WRAPPER_PRESENT")
    if final.count("flash_attn_func(Qnn, Knn, Vnn)") != 0:
        raise RuntimeError("PROGRAM077_LEGACY_FLASH_CALL_REMAINS")
    if final.count("_program077_qwen3_cached_attention(Qnn, Knn, Vnn") != 1:
        raise RuntimeError("PROGRAM077_PATCHED_CALL_COUNT_MISMATCH")
    compile(final, str(qwen3_path), "exec")
    return {
        "path": str(qwen3_path),
        "installed": installed,
        "before_sha256": before_hash,
        "after_sha256": hashlib.sha256(final.encode("utf-8")).hexdigest(),
        "expected_original_sha256": expected_original_sha256,
        "helper_marker_count": final.count(marker),
        "call_replacements": replacements,
        "legacy_flash_calls_after": final.count("flash_attn_func(Qnn, Knn, Vnn)"),
        "generic_flash_wrapper": False,
    }


def write_qwen3_official_kv_probe(path: Path, program_no: int) -> None:
    path.write_text(
        bootstrap_source(program_no)
        + r'''
import json
import torch
import torch.nn.functional as F
import unsloth.models.qwen3 as qwen3

fn = getattr(qwen3, "_program077_qwen3_cached_attention", None)
if not callable(fn):
    raise RuntimeError("PROGRAM077_QWEN3_OFFICIAL_HELPER_MISSING")
backend = getattr(fn, "__program077_backend__", None)
if backend != "official_style_kv_cache":
    raise RuntimeError(f"PROGRAM077_QWEN3_BACKEND={backend}")

torch.manual_seed(47047)
device = torch.device("cuda:0")
dtype = torch.bfloat16
reports = []


def explicit_reference(q, k, v, *, mask=None, sliding_window=None, batch_one=False):
    q_t = q.transpose(1, 2)
    k_t = k.transpose(1, 2)
    v_t = v.transpose(1, 2)
    if sliding_window is not None and k_t.shape[-2] > sliding_window:
        start = k_t.shape[-2] - sliding_window
        k_t = k_t[:, :, start:, :]
        v_t = v_t[:, :, start:, :]
        if mask is not None:
            mask = mask[..., start:]
    if mask is not None:
        if mask.dim() == 2:
            mask = mask[:, None, None, :].to(torch.bool)
        elif mask.dim() == 4 and mask.dtype != torch.bool:
            mask = mask.eq(0)
        if mask.shape[-1] > k_t.shape[-2]:
            mask = mask[..., -k_t.shape[-2]:]
    groups = q_t.shape[1] // k_t.shape[1]
    if groups != 1:
        k_t = k_t.repeat_interleave(groups, dim=1)
        v_t = v_t.repeat_interleave(groups, dim=1)
    if batch_one:
        scores = torch.matmul(q_t * (q_t.shape[-1] ** -0.5), k_t.transpose(-2, -1))
        probs = torch.softmax(scores, dim=-1, dtype=torch.float32).to(q_t.dtype)
        out = torch.matmul(probs, v_t)
    else:
        out = F.scaled_dot_product_attention(
            q_t,
            k_t,
            v_t,
            attn_mask=mask,
            dropout_p=0.0,
            is_causal=(mask is None and q_t.shape[-2] == k_t.shape[-2]),
        )
    return out.transpose(1, 2).contiguous()


cases = [
    {"name": "batched_gqa_decode", "batch": 4, "q_heads": 32, "kv_heads": 8, "q_len": 1, "kv_len": 17},
    {"name": "batched_equal_heads_causal", "batch": 4, "q_heads": 8, "kv_heads": 8, "q_len": 3, "kv_len": 3},
    {"name": "batch_one_explicit", "batch": 1, "q_heads": 32, "kv_heads": 8, "q_len": 1, "kv_len": 17},
    {"name": "sliding_window", "batch": 4, "q_heads": 32, "kv_heads": 8, "q_len": 1, "kv_len": 17, "sliding_window": 8},
    {"name": "batched_masked_decode", "batch": 4, "q_heads": 32, "kv_heads": 8, "q_len": 1, "kv_len": 17, "masked": True},
]
for case in cases:
    q = torch.randn(case["batch"], case["q_len"], case["q_heads"], 16, device=device, dtype=dtype)
    k = torch.randn(case["batch"], case["kv_len"], case["kv_heads"], 16, device=device, dtype=dtype)
    v = torch.randn(case["batch"], case["kv_len"], case["kv_heads"], 16, device=device, dtype=dtype)
    mask = None
    if case.get("masked"):
        mask = torch.ones(case["batch"], case["kv_len"], device=device, dtype=torch.long)
        mask[:, :2] = 0
    window = case.get("sliding_window")
    actual = fn(q, k, v, attention_mask=mask, sliding_window=window)
    expected = explicit_reference(
        q,
        k,
        v,
        mask=mask,
        sliding_window=window,
        batch_one=(case["batch"] == 1),
    )
    torch.cuda.synchronize()
    if actual.shape != expected.shape or not torch.isfinite(actual).all():
        raise RuntimeError(
            f"PROGRAM077_QWEN3_BAD_OUTPUT case={case['name']} actual={actual.shape} expected={expected.shape}"
        )
    max_abs_error = float((actual.float() - expected.float()).abs().max().cpu())
    if max_abs_error > 0.03:
        raise RuntimeError(
            f"PROGRAM077_QWEN3_PARITY_FAILED case={case['name']} max_abs_error={max_abs_error}"
        )
    reports.append({
        "name": case["name"],
        "batch": case["batch"],
        "q_heads": case["q_heads"],
        "kv_heads": case["kv_heads"],
        "q_len": case["q_len"],
        "kv_len": case["kv_len"],
        "sliding_window": window,
        "masked": bool(case.get("masked")),
        "shape": list(actual.shape),
        "max_abs_error": max_abs_error,
    })
print("PROGRAM077_QWEN3_OFFICIAL_KV_PROBE_OK=" + json.dumps({
    "backend": backend,
    "torch": torch.__version__,
    "device": torch.cuda.get_device_name(0),
    "cases": reports,
}, sort_keys=True))
''',
        encoding="utf-8",
    )

def bootstrap_source(program_no: int) -> str:
    return f'''import os\nimport sys\nfrom pathlib import Path\nsource_root = Path(os.environ["PROGRAM{program_no:03d}_LEGACY_ROOT"]).resolve()\noverlay_root = Path(os.environ["PROGRAM{program_no:03d}_OVERLAY_ROOT"]).resolve()\nclean = []\nfor item in sys.path:\n    if not item:\n        clean.append(item)\n        continue\n    try:\n        resolved = Path(item).resolve()\n    except OSError:\n        continue\n    if resolved == source_root or source_root in resolved.parents or resolved == overlay_root:\n        continue\n    clean.append(item)\nsys.path[:] = [str(overlay_root)] + clean\n'''


def write_import_probe(path: Path, program_no: int) -> None:
    path.write_text(
        bootstrap_source(program_no)
        + f'''
import importlib
import json
from pathlib import Path

# Match production import order: native Torch/Triton first, then Unsloth before
# Transformers/PEFT/TRL so Unsloth can apply its patches deterministically.
modules = {{}}
for name in ["torch", "triton"]:
    module = importlib.import_module(name)
    modules[name] = {{
        "version": getattr(module, "__version__", "unknown"),
        "file": str(Path(module.__file__).resolve()) if getattr(module, "__file__", None) else None,
    }}
from unsloth import FastLanguageModel, UnslothTrainer, UnslothTrainingArguments
import unsloth
for name in ["transformers", "datasets", "peft", "accelerate", "bitsandbytes", "trl"]:
    module = importlib.import_module(name)
    modules[name] = {{
        "version": getattr(module, "__version__", "unknown"),
        "file": str(Path(module.__file__).resolve()) if getattr(module, "__file__", None) else None,
    }}
import torch
source_root = Path(os.environ["PROGRAM{program_no:03d}_LEGACY_ROOT"]).resolve()
overlay_root = Path(os.environ["PROGRAM{program_no:03d}_OVERLAY_ROOT"]).resolve()
for anchor in ["torch", "triton"]:
    anchor_path = Path(modules[anchor]["file"]).resolve()
    if anchor_path == source_root or source_root in anchor_path.parents or anchor_path == overlay_root or overlay_root in anchor_path.parents:
        raise RuntimeError(f"{{anchor}} was imported from a legacy overlay: {{anchor_path}}")
visible = torch.cuda.device_count()
if visible < 4:
    raise RuntimeError(f"Expected four visible GPUs, found {{visible}}")
optional = {{}}
for name in ["flash_attn", "xformers", "cut_cross_entropy", "hf_transfer", "msgspec", "sentencepiece"]:
    try:
        module = importlib.import_module(name)
        optional[name] = {{"ok": True, "version": getattr(module, "__version__", "unknown"), "file": str(Path(module.__file__).resolve()) if getattr(module, "__file__", None) else None}}
    except Exception as exc:
        optional[name] = {{"ok": False, "error": f"{{type(exc).__name__}}: {{exc}}"}}
print("PROGRAM{program_no:03d}_IMPORT_PROBE=" + json.dumps({{
    "python": str(Path(sys.executable).resolve()),
    "python_version": sys.version.split()[0],
    "modules": modules,
    "unsloth_version": getattr(unsloth, "__version__", "unknown"),
    "unsloth_file": str(Path(unsloth.__file__).resolve()),
    "optional": optional,
    "cuda_version": torch.version.cuda,
    "device_count": visible,
    "device_names": [torch.cuda.get_device_name(i) for i in range(visible)],
    "required_symbols": {{
        "FastLanguageModel": FastLanguageModel is not None,
        "UnslothTrainer": UnslothTrainer is not None,
        "UnslothTrainingArguments": UnslothTrainingArguments is not None,
    }},
}}, sort_keys=True))
''',
        encoding="utf-8",
    )


def parse_missing_module(stderr: str) -> str | None:
    matches = re.findall(r"ModuleNotFoundError: No module named ['\"]([^'\"]+)['\"]", stderr)
    return matches[-1].split(".", 1)[0] if matches else None


def build_and_probe_profile(
    profile_name: str,
    initial_distributions: list[str],
    source_root: Path,
    by_name: dict[str, Path],
    module_to_dist: dict[str, str],
    base_env: dict[str, str],
    probe_path: Path,
) -> tuple[dict | None, dict]:
    overlay_root = PROFILE_ROOT / profile_name
    if overlay_root.exists():
        shutil.rmtree(overlay_root)
    overlay_root.mkdir(parents=True, exist_ok=True)
    copied_distributions: list[str] = []
    copied_entries: list[str] = []
    errors: list[dict] = []

    for dist in initial_distributions:
        try:
            copied_entries.extend(copy_distribution(source_root, overlay_root, dist, by_name))
            copied_distributions.append(canonical_name(dist))
        except FileNotFoundError:
            if canonical_name(dist) in {"unsloth", "unsloth-zoo", "transformers", "peft", "accelerate", "datasets", "trl"}:
                raise
    patches = patch_legacy_transformers(overlay_root)

    for attempt in range(1, 25):
        env = base_env.copy()
        env.update({
            "PROGRAM077_LEGACY_ROOT": str(source_root),
            "PROGRAM077_OVERLAY_ROOT": str(overlay_root),
            "PYTHONPATH": sanitize_pythonpath(base_env.get("PYTHONPATH", ""), source_root, overlay_root),
        })
        result = subprocess.run(
            [sys.executable, str(probe_path)],
            env=env,
            cwd=str(WORK_ROOT),
            text=True,
            capture_output=True,
        )
        if result.stdout:
            print(result.stdout, end="")
        if result.returncode == 0:
            marker = next((line for line in result.stdout.splitlines() if line.startswith("PROGRAM077_IMPORT_PROBE=")), None)
            if marker is None:
                raise RuntimeError(f"Profile {profile_name} succeeded without an import marker")
            info = json.loads(marker.split("=", 1)[1])
            info.update({
                "profile": profile_name,
                "overlay_root": str(overlay_root),
                "runtime_pythonpath": env["PYTHONPATH"],
                "copied_distributions": copied_distributions,
                "copied_entries": copied_entries,
                "patches": patches,
            })
            return info, {"profile": profile_name, "success": True, "attempts": attempt}

        missing = parse_missing_module(result.stderr)
        errors.append({
            "attempt": attempt,
            "returncode": result.returncode,
            "missing": missing,
            "stderr_tail": " | ".join(result.stderr.strip().splitlines()[-10:]),
        })
        if not missing or missing in NATIVE_BINARY_ANCHORS:
            break
        dist = module_to_dist.get(missing)
        if not dist or dist in copied_distributions:
            break
        try:
            copied_entries.extend(copy_distribution(source_root, overlay_root, dist, by_name))
            copied_distributions.append(dist)
            patches.extend(patch_legacy_transformers(overlay_root))
            print(f"PROGRAM077_PROFILE_RETRY profile={profile_name} missing={missing} copied={dist}")
        except Exception as exc:
            errors.append({"attempt": attempt, "copy_error": f"{type(exc).__name__}: {exc}"})
            break

    return None, {
        "profile": profile_name,
        "success": False,
        "copied_distributions": copied_distributions,
        "copied_entries": copied_entries,
        "patches": patches,
        "errors": errors,
    }


rerun_mode = env_flag("KAGGLE_IS_COMPETITION_RERUN")
challenge_names = (
    ["arc-agi_test_challenges.json", "arc_agi_test_challenges.json"]
    if rerun_mode
    else ["arc-agi_evaluation_challenges.json", "arc_agi_evaluation_challenges.json"]
)
challenge_path = resolve_named_file(challenge_names)
solution_path = None if rerun_mode else resolve_named_file(
    ["arc-agi_evaluation_solutions.json", "arc_agi_evaluation_solutions.json"]
)
if not MODEL_PATH.is_dir():
    raise FileNotFoundError(f"Model directory is missing: {MODEL_PATH}")

ptxas_path = resolve_ptxas()
ptxas_version = subprocess.check_output([str(ptxas_path), "--version"], text=True, stderr=subprocess.STDOUT).splitlines()[0]
legacy_roots = discover_legacy_roots()
runtime_selection_attempts: list[dict] = []
selected_runtime_family: str | None = None
legacy_root: Path | None = None
by_name: dict[str, Path] | None = None
module_to_dist: dict[str, str] | None = None
available_versions: dict[str, list[str]] | None = None
pinned_source_report: dict | None = None

for family_name, family_lock in RUNTIME_FAMILIES:
    for candidate_root in legacy_roots:
        try:
            candidate_by_name, candidate_module_map, candidate_versions = distribution_index(
                candidate_root, family_lock, family_name
            )
            PINNED_RUNTIME_DISTRIBUTIONS = family_lock
            candidate_report = verify_pinned_runtime(candidate_root, candidate_by_name)
        except Exception as exc:
            runtime_selection_attempts.append({
                "runtime_family": family_name,
                "source_root": str(candidate_root),
                "success": False,
                "error": f"{type(exc).__name__}: {exc}",
            })
            continue
        selected_runtime_family = family_name
        legacy_root = candidate_root
        by_name = candidate_by_name
        module_to_dist = candidate_module_map
        available_versions = candidate_versions
        pinned_source_report = candidate_report
        runtime_selection_attempts.append({
            "runtime_family": family_name,
            "source_root": str(candidate_root),
            "success": True,
            "versions": {
                name: lock["version"] for name, lock in sorted(family_lock.items())
            },
        })
        break
    if selected_runtime_family is not None:
        break

if (
    selected_runtime_family is None
    or legacy_root is None
    or by_name is None
    or module_to_dist is None
    or available_versions is None
    or pinned_source_report is None
):
    raise RuntimeError(
        "PROGRAM077_NO_VERIFIED_RUNTIME_FAMILY="
        + json.dumps({
            "roots": [str(root) for root in legacy_roots],
            "attempts": runtime_selection_attempts,
        }, sort_keys=True)
    )

print("PROGRAM077_RUNTIME_FAMILY_SELECTED=" + json.dumps({
    "runtime_family": selected_runtime_family,
    "source_root": str(legacy_root),
    "versions": {
        name: lock["version"] for name, lock in sorted(PINNED_RUNTIME_DISTRIBUTIONS.items())
    },
    "root_count": len(legacy_roots),
    "attempt_count": len(runtime_selection_attempts),
}, sort_keys=True))
print("PROGRAM077_PINNED_RUNTIME_SOURCE_OK=" + json.dumps({
    "runtime_family": selected_runtime_family,
    "source_root": str(legacy_root),
    "available_versions": {
        name: available_versions.get(name, [])
        for name in sorted(PINNED_RUNTIME_DISTRIBUTIONS)
    },
    "locked": pinned_source_report,
}, sort_keys=True))
PROFILE_ROOT.mkdir(parents=True, exist_ok=True)

base_env = os.environ.copy()
base_env.update({
    "TRITON_PTXAS_PATH": str(ptxas_path),
    "PYTHONHASHSEED": PYTHON_HASH_SEED,
    "PROGRAM077_BASE_SEED": str(BASE_SEED),
    "PROGRAM077_SEED_POLICY": "rng_only_no_backend_change",
    "UNSLOTH_DISABLE_STATISTICS": "1",
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
})

probe_path = WORK_ROOT / "program077_import_probe.py"
write_import_probe(probe_path, PROGRAM_NO)

profiles = [
    # Program 073 intentionally has no native-core fallback. A fallback could
    # silently reintroduce the newer Accelerate/PEFT/bitsandbytes combination.
    (
        "legacy_python_stack_pinned",
        [
            "unsloth", "unsloth-zoo", "transformers", "peft", "accelerate",
            "datasets", "trl", "huggingface-hub", "bitsandbytes",
        ],
    ),
]

selected = None
profile_reports = []
for profile_name, initial in profiles:
    try:
        info, report = build_and_probe_profile(
            profile_name, initial, legacy_root, by_name, module_to_dist, base_env, probe_path
        )
    except Exception as exc:
        info = None
        report = {"profile": profile_name, "success": False, "build_error": f"{type(exc).__name__}: {exc}"}
    profile_reports.append(report)
    print("PROGRAM077_PROFILE_REPORT=" + json.dumps(report, sort_keys=True))
    if info is not None:
        selected = info
        break

if selected is None:
    raise RuntimeError(
        "PROGRAM077_RUNTIME_PROFILE_FAILED: the exact pinned legacy Python-stack profile could not "
        "import the complete Unsloth runtime. No native-core fallback is permitted. "
        "Aborting before every GPU probe and model loading. Reports="
        + json.dumps(profile_reports, sort_keys=True)
    )

imported_pinned_versions = {}
for module_name, lock in sorted(PINNED_RUNTIME_DISTRIBUTIONS.items()):
    actual = selected.get("modules", {}).get(module_name, {}).get("version")
    imported_pinned_versions[module_name] = actual
    if actual != lock["version"]:
        raise RuntimeError(
            "PROGRAM077_PINNED_IMPORT_VERSION_MISMATCH="
            + json.dumps({
                "module": module_name,
                "expected": lock["version"],
                "actual": actual,
                "module_info": selected.get("modules", {}).get(module_name),
            }, sort_keys=True)
        )

overlay_root_path = Path(selected["overlay_root"])
overlay_by_name, _, overlay_available_versions = distribution_index(
    overlay_root_path, PINNED_RUNTIME_DISTRIBUTIONS, selected_runtime_family
)
pinned_overlay_report = verify_pinned_runtime(overlay_root_path, overlay_by_name)
for module_name in sorted(PINNED_RUNTIME_DISTRIBUTIONS):
    source_fingerprint = pinned_source_report[module_name]["record_tree_sha256"]
    overlay_fingerprint = pinned_overlay_report[module_name]["record_tree_sha256"]
    if source_fingerprint != overlay_fingerprint:
        raise RuntimeError(
            "PROGRAM077_PINNED_OVERLAY_FINGERPRINT_MISMATCH="
            + json.dumps({
                "module": module_name,
                "source": source_fingerprint,
                "overlay": overlay_fingerprint,
            }, sort_keys=True)
        )
print("PROGRAM077_PINNED_RUNTIME_OVERLAY_OK=" + json.dumps({
    "imported_versions": imported_pinned_versions,
    "overlay_available_versions": {
        name: overlay_available_versions.get(name, [])
        for name in sorted(PINNED_RUNTIME_DISTRIBUTIONS)
    },
    "locked": pinned_overlay_report,
}, sort_keys=True))

qwen3_patch_report = install_qwen3_official_kv_cache_patch(Path(selected["overlay_root"]))
print("PROGRAM077_QWEN3_OFFICIAL_KV_PATCH_OK=" + json.dumps(qwen3_patch_report, sort_keys=True))

selected_env = base_env.copy()
selected_env.update({
    "PROGRAM077_LEGACY_ROOT": str(legacy_root),
    "PROGRAM077_OVERLAY_ROOT": selected["overlay_root"],
    "PYTHONPATH": selected["runtime_pythonpath"],
})

qwen3_probe_path = WORK_ROOT / "program077_qwen3_official_kv_probe.py"
write_qwen3_official_kv_probe(qwen3_probe_path, PROGRAM_NO)
qwen3_probe_env = selected_env.copy()
qwen3_probe_env.update({"CUDA_VISIBLE_DEVICES": "0"})
qwen3_probe_result = subprocess.run(
    [sys.executable, str(qwen3_probe_path)],
    env=qwen3_probe_env,
    cwd=str(WORK_ROOT),
    text=True,
    capture_output=True,
    timeout=180,
)
if qwen3_probe_result.stdout:
    print(qwen3_probe_result.stdout, end="")
if qwen3_probe_result.returncode != 0:
    if qwen3_probe_result.stderr:
        print(qwen3_probe_result.stderr, file=sys.stderr, end="")
    raise RuntimeError(
        f"PROGRAM077_QWEN3_OFFICIAL_KV_PROBE_FAILED exit_code={qwen3_probe_result.returncode}"
    )
qwen3_probe_marker = next(
    (
        line
        for line in qwen3_probe_result.stdout.splitlines()
        if line.startswith("PROGRAM077_QWEN3_OFFICIAL_KV_PROBE_OK=")
    ),
    None,
)
if qwen3_probe_marker is None:
    raise RuntimeError("PROGRAM077_QWEN3_OFFICIAL_KV_PROBE_MARKER_MISSING")
qwen3_probe_report = json.loads(qwen3_probe_marker.split("=", 1)[1])

triton_probe_path = WORK_ROOT / "program077_triton_probe.py"
triton_probe_path.write_text(
    bootstrap_source(PROGRAM_NO)
    + r'''
import torch
import triton
import triton.language as tl

@triton.jit
def copy_kernel(x_ptr, y_ptr, n_elements: tl.constexpr, BLOCK_SIZE: tl.constexpr):
    offsets = tl.arange(0, BLOCK_SIZE)
    mask = offsets < n_elements
    values = tl.load(x_ptr + offsets, mask=mask)
    tl.store(y_ptr + offsets, values, mask=mask)

x = torch.arange(16, device="cuda:0", dtype=torch.float32)
y = torch.empty_like(x)
copy_kernel[(1,)](x, y, n_elements=16, BLOCK_SIZE=16)
torch.cuda.synchronize()
if not torch.equal(x, y):
    raise RuntimeError("Triton probe produced an incorrect result")
print("PROGRAM077_TRITON_PROBE_OK")
''',
    encoding="utf-8",
)
triton_result = subprocess.run(
    [sys.executable, str(triton_probe_path)],
    env=selected_env,
    cwd=str(WORK_ROOT),
    text=True,
    capture_output=True,
)
if triton_result.stdout:
    print(triton_result.stdout, end="")
if triton_result.returncode != 0:
    if triton_result.stderr:
        print(triton_result.stderr, file=sys.stderr, end="")
    raise RuntimeError(f"PROGRAM077_TRITON_PROBE_FAILED exit_code={triton_result.returncode}")

run_plan = {
    "program_no": PROGRAM_NO,
    "rerun_mode": rerun_mode,
    "challenge_path": str(challenge_path),
    "solution_path": str(solution_path) if solution_path else None,
    "model_path": str(MODEL_PATH),
    "ptxas_path": str(ptxas_path),
    "ptxas_version": ptxas_version,
    "legacy_root": str(legacy_root),
    "overlay_root": selected["overlay_root"],
    "runtime_pythonpath": selected["runtime_pythonpath"],
    "python_executable": str(Path(sys.executable).resolve()),
    "python_version": selected["python_version"],
    "runtime_profile": selected["profile"],
    "runtime_family": selected_runtime_family,
    "runtime_selection_attempts": runtime_selection_attempts,
    "runtime_info": selected,
    "pinned_runtime_versions": {
        name: lock["version"] for name, lock in sorted(PINNED_RUNTIME_DISTRIBUTIONS.items())
    },
    "pinned_runtime_source": pinned_source_report,
    "pinned_runtime_overlay": pinned_overlay_report,
    "pinned_runtime_imported_versions": imported_pinned_versions,
    "pinned_runtime_available_versions": {
        name: available_versions.get(name, [])
        for name in sorted(PINNED_RUNTIME_DISTRIBUTIONS)
    },
    "qwen3_official_kv_patch": qwen3_patch_report,
    "qwen3_official_kv_probe": qwen3_probe_report,
    "seed_contract": SEED_CONTRACT,
    "global_end_time": global_end_time,
    "hard_end_time": hard_end_time,
    "hard_runtime_limit_seconds": HARD_RUNTIME_LIMIT_SECONDS,
    "finalization_reserve_seconds": FINALIZATION_RESERVE_SECONDS,
    "notebook_start_time": NOTEBOOK_START_TIME,
    "recovery_policy": {
        "name": "normal_cuda_program024_primary_only",
        "max_deep_tasks": 0,
        "deep_retry_wall_limit_seconds": 0,
        "selection_policy": "exact_kgmon_two_guesses_no_posthoc_change",
        "local_evaluation_policy": "diagnostic_only_normalized_never_submission_gate",
        "model_smoke_count": 1,
    },
}
Path("program077_run_plan.json").write_text(json.dumps(run_plan, indent=2, sort_keys=True), encoding="utf-8")

print(
    "PROGRAM077_PREFLIGHT_OK "
    f"profile={selected['profile']} python={selected['python_version']} "
    f"torch={selected['modules']['torch']['version']} triton={selected['modules']['triton']['version']} "
    f"transformers={selected['modules']['transformers']['version']} unsloth={selected['unsloth_version']} "
    f"gpus={selected['device_count']} "
    f"runtime_family={selected_runtime_family} "
    f"runtime_lock=accelerate{PINNED_RUNTIME_DISTRIBUTIONS['accelerate']['version']}_"
    f"peft{PINNED_RUNTIME_DISTRIBUTIONS['peft']['version']}_"
    f"bnb{PINNED_RUNTIME_DISTRIBUTIONS['bitsandbytes']['version']} "
    f"seed_policy=rng_only base_seed={BASE_SEED} python_hash_seed={PYTHON_HASH_SEED} "
    f"qwen3_attention={qwen3_probe_report['backend']} "
    f"challenge={challenge_path}"
)


In [ ]:
# Remove TensorFlow only after the complete offline runtime profile and Triton kernel have passed.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "tensorflow"], check=False)


In [ ]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

In [ ]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
        print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

In [ ]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


# Minimal performance patch: preserve the baseline beam set and ranking, but transfer
# only the 12 ARC-token NLL values to CPU instead of every Qwen vocabulary logit.
_ARC_TOKEN_ID_CACHE = {}


def _arc_token_ids(device):
    key = str(device)
    token_ids = _ARC_TOKEN_ID_CACHE.get(key)
    if token_ids is None:
        token_ids = torch.tensor(ARC_TOKENS, dtype=torch.long, device=device)
        _ARC_TOKEN_ID_CACHE[key] = token_ids
    return token_ids


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    # Algebraically identical to: scores - logits.float().cpu().log_softmax(-1),
    # restricted to the same ARC_TOKENS used by the baseline DFS loop.
    logits_f = logits.float()
    token_ids = _arc_token_ids(logits.device)
    arc_logits = logits_f.index_select(-1, token_ids)
    nll = (
        torch.as_tensor(scores, dtype=torch.float32, device=logits.device).view(n, 1)
        + torch.logsumexp(logits_f, dim=-1, keepdim=True)
        - arc_logits
    ).cpu()

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for token_idx, t in enumerate(ARC_TOKENS):
            score = nll[i, token_idx].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)

    # Keep logits on GPU and gather only the target-token scores. KV cache is not
    # consumed by teacher-forced scoring, so disabling it removes redundant writes.
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=False)
    batch_logits = outputs.logits.float()
    batch_log_norm = torch.logsumexp(batch_logits, dim=-1)
    result = []
    for row_id, (query_tokens, answer_tokens) in enumerate(zip(batch_query_tokens, batch_answer_tokens)):
        query_length = len(query_tokens)
        answer_length = len(answer_tokens)
        positions = torch.arange(
            query_length - 1,
            query_length - 1 + answer_length,
            device=model.device,
        )
        target_tokens = torch.tensor(answer_tokens, device=model.device, dtype=torch.long)
        answer_log_probs = (
            batch_logits[row_id, positions, target_tokens]
            - batch_log_norm[row_id, positions]
        )
        result.append(-answer_log_probs.sum().item())
    return result


def worker(rank, queue, end_time, test_path):

    if not os.path.isfile(test_path):
        raise FileNotFoundError(f"Resolved ARC challenge file is missing: {test_path}")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim="adamw_torch",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
    )

    max_seq_length = 8192

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name="/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)

    for name, param in model.named_parameters():
        if param.dtype == torch.float32:
            param.data = param.data.to(torch.bfloat16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while True:

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break
        
        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            # 0: permute x 2
            # 4: rot90.rot90.permute x 2
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 2: permute.rot90 x 2
            # 6: rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            # 8: transpose.permute x 2
            # 12: transpose.rot90.rot90.permute x 2
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 10: transpose.rot90.permute x 2
            # 14: transpose.rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        with torch.inference_mode():
                
            known_scores = {}

            for subkeys in batches:

                spend_time = time.time() - start_time
                if spend_time > 1200 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

In [ ]:
%%writefile starter.py
import hashlib
import json
import os
import pickle
import random
import shutil
import sys
import time
from pathlib import Path


def sanitize_runtime_path():
    source_raw = os.environ.get("PROGRAM077_LEGACY_ROOT")
    overlay_raw = os.environ.get("PROGRAM077_OVERLAY_ROOT")
    if not source_raw or not overlay_raw:
        raise RuntimeError("Program No. 077 runtime path variables are missing")
    source_root = Path(source_raw).resolve()
    overlay_root = Path(overlay_raw).resolve()
    clean = []
    for item in sys.path:
        if not item:
            clean.append(item)
            continue
        try:
            resolved = Path(item).resolve()
        except OSError:
            continue
        if resolved == source_root or source_root in resolved.parents:
            continue
        if resolved == overlay_root:
            continue
        clean.append(item)
    sys.path[:] = [str(overlay_root)] + clean


sanitize_runtime_path()

import numpy as np
import torch
import argparse
import torch.multiprocessing as mp


STARTER_LOCK = Path("/kaggle/working/program077_starter.lock")
SEED_PROBE_TEXT = "arc-program-seed-probe-v1"


def sha256_bytes(value):
    return hashlib.sha256(value).hexdigest()


def seed_contract():
    raw_seed = os.environ.get("PROGRAM077_BASE_SEED")
    if raw_seed is None:
        raise RuntimeError("PROGRAM077_BASE_SEED is missing")
    try:
        base_seed = int(raw_seed)
    except ValueError as exc:
        raise RuntimeError(f"PROGRAM077_BASE_SEED is invalid: {raw_seed!r}") from exc
    python_hash_seed = os.environ.get("PYTHONHASHSEED")
    if python_hash_seed != "0":
        raise RuntimeError(
            f"PROGRAM077_PYTHONHASHSEED_INVALID expected='0' actual={python_hash_seed!r}"
        )
    policy = os.environ.get("PROGRAM077_SEED_POLICY")
    if policy != "rng_only_no_backend_change":
        raise RuntimeError(f"PROGRAM077_SEED_POLICY_INVALID={policy!r}")
    return base_seed, python_hash_seed, policy


def rng_state_report(stage, rank, task_id=None):
    base_seed, python_hash_seed, policy = seed_contract()
    run_label = os.environ.get("PROGRAM077_RUN_LABEL", "unknown")
    random.seed(base_seed)
    np.random.seed(base_seed)
    torch.manual_seed(base_seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(base_seed)

    python_state = pickle.dumps(random.getstate(), protocol=4)
    numpy_state = pickle.dumps(np.random.get_state(), protocol=4)
    torch_cpu_state = torch.get_rng_state().cpu().numpy().tobytes()
    cuda_state_hashes = []
    if torch.cuda.is_available():
        for state in torch.cuda.get_rng_state_all():
            cuda_state_hashes.append(sha256_bytes(state.cpu().numpy().tobytes()))

    return {
        "stage": stage,
        "run_label": run_label,
        "rank": rank,
        "task_id": task_id,
        "base_seed": base_seed,
        "python_hash_seed": python_hash_seed,
        "seed_policy": policy,
        "hash_probe": hash(SEED_PROBE_TEXT),
        "python_rng_sha256": sha256_bytes(python_state),
        "numpy_rng_sha256": sha256_bytes(numpy_state),
        "torch_cpu_rng_sha256": sha256_bytes(torch_cpu_state),
        "torch_cuda_rng_sha256": cuda_state_hashes,
        "torch_initial_seed": int(torch.initial_seed()),
        "deterministic_algorithms_enabled": bool(torch.are_deterministic_algorithms_enabled()),
        "cudnn_deterministic": bool(torch.backends.cudnn.deterministic),
        "cudnn_benchmark": bool(torch.backends.cudnn.benchmark),
        "cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    }


class SeededQueueProxy:
    def __init__(self, queue, rank):
        self._queue = queue
        self._rank = rank

    def get(self, *args, **kwargs):
        item = self._queue.get(*args, **kwargs)
        if item is not None:
            report = rng_state_report("task_dequeue", self._rank, str(item))
            print("PROGRAM077_TASK_SEED_READY=" + json.dumps(report, sort_keys=True), flush=True)
        return item


def acquire_single_starter_lock():
    try:
        fd = os.open(STARTER_LOCK, os.O_CREAT | os.O_EXCL | os.O_WRONLY, 0o600)
    except FileExistsError as exc:
        owner = STARTER_LOCK.read_text(encoding="utf-8", errors="replace") if STARTER_LOCK.exists() else "unknown"
        raise RuntimeError(f"PROGRAM077_DUPLICATE_STARTER_BLOCKED existing={owner}") from exc
    try:
        os.write(fd, f"pid={os.getpid()} started={time.time():.6f}\n".encode("utf-8"))
    finally:
        os.close(fd)
    print(f"PROGRAM077_STARTER_LOCK_ACQUIRED pid={os.getpid()} path={STARTER_LOCK}", flush=True)


def release_single_starter_lock():
    try:
        STARTER_LOCK.unlink()
    except FileNotFoundError:
        pass


def local_worker(rank, queue, end_time, test_path):
    run_label = os.environ["PROGRAM077_RUN_LABEL"]
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)
    torch.set_default_device("cpu")
    worker_seed_report = rng_state_report("worker_init", rank)
    print("PROGRAM077_WORKER_SEED_READY=" + json.dumps(worker_seed_report, sort_keys=True), flush=True)

    if rank > 0:
        previous_marker = Path(f"/kaggle/worker{rank - 1}")
        while not previous_marker.exists():
            if time.time() >= end_time:
                raise TimeoutError(
                    f"Rank {rank} reached the {run_label} deadline while waiting for {previous_marker}"
                )
            time.sleep(2)

    # Import only after this process receives its Primary seed state.
    if run_label != "primary":
        raise RuntimeError(f"PROGRAM077_UNKNOWN_RUN_LABEL={run_label!r}")
    from arc_solver import worker

    Path(f"/kaggle/worker{rank}").write_text(
        json.dumps({"run_label": run_label, "rank": rank, "pid": os.getpid(), "gpu": rank}, sort_keys=True),
        encoding="utf-8",
    )
    print(f"[Rank {rank}] start! run_label={run_label} pid={os.getpid()} gpu={rank}", flush=True)
    worker(rank, SeededQueueProxy(queue, rank), end_time, test_path)
    print(f"[Rank {rank}] done! run_label={run_label} pid={os.getpid()} gpu={rank}", flush=True)


def clean_runtime_state():
    output_dir = Path("/kaggle/inference_outputs")
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    for rank in range(4):
        marker = Path(f"/kaggle/worker{rank}")
        if marker.exists():
            marker.unlink()


def load_selected_keys(task_file, challenge_keys):
    if task_file is None:
        return sorted(challenge_keys)
    payload = json.loads(Path(task_file).read_text(encoding="utf-8"))
    if not isinstance(payload, list) or not payload:
        raise ValueError("Retry task file must be a non-empty JSON list")
    if any(not isinstance(task_id, str) for task_id in payload):
        raise TypeError("Every retry task id must be a string")
    if len(payload) != len(set(payload)):
        raise ValueError("Retry task file contains duplicate task ids")
    unknown = sorted(set(payload) - set(challenge_keys))
    if unknown:
        raise KeyError(f"Retry task file contains unknown ids: {unknown}")
    return payload


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, required=True)
    parser.add_argument("--challenge-path", type=str, required=True)
    parser.add_argument("--task-file", type=str)
    parser.add_argument("--run-label", choices=("primary",), required=True)
    args = parser.parse_args()
    os.environ["PROGRAM077_RUN_LABEL"] = args.run_label
    base_seed, python_hash_seed, seed_policy = seed_contract()
    acquire_single_starter_lock()
    try:
        test_path = str(Path(args.challenge_path).resolve())
        if not os.path.isfile(test_path):
            raise FileNotFoundError(f"Challenge file not found: {test_path}")
        with open(test_path, "r", encoding="utf-8") as handle:
            data = json.load(handle)
        if not isinstance(data, dict) or not data:
            raise ValueError("Challenge JSON must be a non-empty object")
        visible_gpus = torch.cuda.device_count()
        device_names = [torch.cuda.get_device_name(index) for index in range(visible_gpus)]
        if visible_gpus != 4 or device_names != ["NVIDIA L4"] * 4:
            raise RuntimeError(
                "PROGRAM077_STARTER_GPU_GATE_FAILED="
                + json.dumps({"device_count": visible_gpus, "device_names": device_names}, sort_keys=True)
            )
        torch_file = Path(torch.__file__).resolve()
        source_root = Path(os.environ["PROGRAM077_LEGACY_ROOT"]).resolve()
        if torch_file == source_root or source_root in torch_file.parents:
            raise RuntimeError(f"Program No. 077 loaded legacy Torch unexpectedly: {torch_file}")
        clean_runtime_state()
        selected_keys = load_selected_keys(args.task_file, data.keys())
        print(
            "PROGRAM077_CHALLENGE_SCOPE="
            + json.dumps({
                "run_label": args.run_label,
                "full_scope": args.task_file is None,
                "selected_task_count": len(selected_keys),
                "selected_tasks": selected_keys,
            }, sort_keys=True),
            flush=True,
        )
        manager = mp.Manager()
        queue = manager.Queue()
        for key in selected_keys:
            queue.put(key)
        for _ in range(4):
            queue.put(None)
        print(
            f"PROGRAM077_STARTER_READY run_label={args.run_label} parent_pid={os.getpid()} "
            f"tasks={len(selected_keys)} gpus={visible_gpus} device_names={device_names} torch={torch_file} "
            f"base_seed={base_seed} python_hash_seed={python_hash_seed} seed_policy={seed_policy}",
            flush=True,
        )
        mp.spawn(local_worker, args=(queue, args.end_time, test_path), nprocs=4)
        print(
            f"PROGRAM077_STARTER_COMPLETE run_label={args.run_label} parent_pid={os.getpid()} "
            f"workers=4 tasks={len(selected_keys)}",
            flush=True,
        )
    finally:
        release_single_starter_lock()


In [ ]:
from __future__ import annotations

import ast
import hashlib
import json
import os
import py_compile
import re
import subprocess
import sys
import time
from pathlib import Path


def normalize_python_source(text: str) -> str:
    return text.replace("\r\n", "\n").replace("\r", "\n").rstrip("\n")


def normalized_source_sha256(path: str) -> str:
    text = Path(path).read_text(encoding="utf-8")
    return hashlib.sha256(normalize_python_source(text).encode("utf-8")).hexdigest()


EXPECTED_PROGRAM024_HASHES = {
    "arc_loader.py": "d01cd56167e534ae156706ef62ab11662fc7b403964614a7a55131940bde970c",
    "arc_decoder.py": "965cfd910777d9bbec9681c6c15c5e2ea3924569734248046f5e58f16cb28222",
    "arc_solver.py": "f9011b4d4549fb688488d67ba7c2dc43184824436d34e1a4c2eae06f69daab73",
}
actual_hashes = {
    name: normalized_source_sha256(name)
    for name in EXPECTED_PROGRAM024_HASHES
}
if actual_hashes != EXPECTED_PROGRAM024_HASHES:
    raise RuntimeError(
        "PROGRAM077_PROGRAM024_SOURCE_MISMATCH="
        + json.dumps({"expected": EXPECTED_PROGRAM024_HASHES, "actual": actual_hashes}, sort_keys=True)
    )
print("PROGRAM077_EXACT_PROGRAM024_SOURCE_HASH_OK=" + json.dumps(actual_hashes, sort_keys=True))

for script in ["arc_loader.py", "arc_decoder.py", "arc_solver.py", "starter.py"]:
    py_compile.compile(script, doraise=True)

solver_source = Path("arc_solver.py").read_text(encoding="utf-8")
decoder_source = Path("arc_decoder.py").read_text(encoding="utf-8")
starter_source = Path("starter.py").read_text(encoding="utf-8")
solver_tree = ast.parse(solver_source, filename="arc_solver.py")
decoder_tree = ast.parse(decoder_source, filename="arc_decoder.py")
ast.parse(starter_source, filename="starter.py")

peft_params = None
for node in ast.walk(solver_tree):
    if isinstance(node, ast.Assign) and any(
        isinstance(target, ast.Name) and target.id == "peft_params"
        for target in node.targets
    ):
        if isinstance(node.value, ast.Call) and isinstance(node.value.func, ast.Name) and node.value.func.id == "dict":
            peft_params = {kw.arg: ast.literal_eval(kw.value) for kw in node.value.keywords}
        else:
            peft_params = ast.literal_eval(node.value)
        break
if peft_params is None:
    raise RuntimeError("PROGRAM077_PEFT_PARAMS_MISSING")
for key, expected in {"r": 256, "lora_alpha": 32, "use_rslora": True}.items():
    if peft_params.get(key) != expected:
        raise RuntimeError(
            f"PROGRAM077_PEFT_PARAM_MISMATCH key={key} actual={peft_params.get(key)!r} expected={expected!r}"
        )

required_patterns = {
    "original training augmentation": r"train_ds\s*=\s*puzzle_ds\.augment\(n=16,\s*shfl_keys=True,\s*seed=1\)",
    "original evaluation augmentation": r"eval_ds\s*=\s*puzzle_ds_multi\.augment\(n=2,\s*seed=2\)",
    "original scoring augmentation": r"aug_dataset\s*=\s*aug_dataset\.augment\(seed=hash\(bk\)\s*%\s*1024\*\*2\)",
    "original DFS window": r"time\.time\(\)\s*-\s*start_time\s*<\s*540\b",
    "original task timeout": r"spend_time\s*>\s*1200\b",
    "blocking queue": r"\bkey\s*=\s*queue\.get\(\)",
    "queue sentinel": r"if\s+key\s+is\s+None\s*:",
}
for label, pattern in required_patterns.items():
    if re.search(pattern, solver_source) is None:
        raise RuntimeError(f"PROGRAM077_BASELINE_INVARIANT_MISSING={label}")


# KGMon is implemented in arc_decoder.py, not arc_solver.py. Verify it structurally
# instead of using a case-sensitive text search in the wrong file.
decoder_functions = {
    node.name: node
    for node in decoder_tree.body
    if isinstance(node, ast.FunctionDef)
}
if "score_kgmon" not in decoder_functions:
    raise RuntimeError("PROGRAM077_KGMON_SCORE_FUNCTION_MISSING")
selection_list_ok = False
run_selection_default_ok = False
for node in decoder_tree.body:
    if isinstance(node, ast.Assign) and any(
        isinstance(target, ast.Name) and target.id == "selection_algorithms"
        for target in node.targets
    ):
        if isinstance(node.value, (ast.List, ast.Tuple)):
            selection_list_ok = any(
                isinstance(item, ast.Name) and item.id == "score_kgmon"
                for item in node.value.elts
            )
    if isinstance(node, ast.ClassDef) and node.name == "ArcDecoder":
        for method in node.body:
            if isinstance(method, ast.FunctionDef) and method.name == "run_selection_algo":
                defaults = method.args.defaults
                run_selection_default_ok = bool(
                    defaults
                    and isinstance(defaults[-1], ast.Name)
                    and defaults[-1].id == "score_kgmon"
                )
if not selection_list_ok:
    raise RuntimeError("PROGRAM077_KGMON_SELECTION_LIST_MISSING")
if not run_selection_default_ok:
    raise RuntimeError("PROGRAM077_KGMON_DEFAULT_SELECTION_MISMATCH")
print("PROGRAM077_KGMON_AST_OK")

for forbidden in [
    "build_factorized_symmetry_product",
    "build_color_descriptors",
    "stable_task_seed",
    "SYMMETRY_PLAN",
    "score_dual_ranker",
    "score_shape_mismatch_only_rescue",
    "replace_structurally_redundant_second",
    "borrowed_seconds",
    "deadline_blocked_tasks",
    "ARC_DFS_FORWARD_BUDGET",
    "while not queue.empty()",
]:
    if forbidden in solver_source:
        raise RuntimeError(f"PROGRAM077_FORBIDDEN_EXPERIMENTAL_CODE={forbidden}")

for marker in [
    "clean_runtime_state()",
    "marker.unlink()",
    "shutil.rmtree(output_dir)",
    "acquire_single_starter_lock()",
    "PROGRAM077_STARTER_COMPLETE",
    "PROGRAM077_WORKER_SEED_READY",
    "PROGRAM077_TASK_SEED_READY",
    "SeededQueueProxy",
    "rng_state_report",
]:
    if marker not in starter_source:
        raise RuntimeError(f"PROGRAM077_STARTER_INVARIANT_MISSING={marker}")

run_plan = json.loads(Path("program077_run_plan.json").read_text(encoding="utf-8"))
challenge_scope_path = Path(run_plan["challenge_path"]).resolve()
challenge_scope_payload = json.loads(challenge_scope_path.read_text(encoding="utf-8"))
if not isinstance(challenge_scope_payload, dict) or not challenge_scope_payload:
    raise RuntimeError(f"PROGRAM077_INVALID_CHALLENGE_SCOPE={challenge_scope_path}")
expected_tasks = set(challenge_scope_payload)
expected_task_count = len(expected_tasks)
if not run_plan["rerun_mode"] and expected_task_count != 120:
    raise RuntimeError(
        "PROGRAM077_VISIBLE_EVALUATION_SCOPE_COUNT_MISMATCH="
        + json.dumps({"expected": 120, "actual": expected_task_count}, sort_keys=True)
    )
print("PROGRAM077_PRECOMPUTE_SCOPE_AUDIT_OK=" + json.dumps({
    "challenge_path": str(challenge_scope_path),
    "rerun_mode": bool(run_plan["rerun_mode"]),
    "expected_task_count": expected_task_count,
}, sort_keys=True))
if not isinstance(EARLY_GPU_REPORT, dict) or EARLY_GPU_REPORT.get("device_names") != ["NVIDIA L4"] * 4:
    raise RuntimeError("PROGRAM077_EARLY_GPU_AUDIT_MISSING_OR_INVALID")
print("PROGRAM077_EARLY_GPU_AUDIT_OK")

if run_plan.get("program_no") != PROGRAM_NO:
    raise RuntimeError(f"PROGRAM077_RUN_PLAN_ID_MISMATCH={run_plan.get('program_no')}")
expected_seed_contract = {
    "base_seed": 42,
    "python_hash_seed": "0",
    "worker_init_reset": True,
    "task_dequeue_reset": True,
    "deterministic_algorithms_enabled": False,
    "backend_policy_changed": False,
}
if run_plan.get("seed_contract") != expected_seed_contract:
    raise RuntimeError(
        "PROGRAM077_SEED_CONTRACT_MISMATCH="
        + json.dumps({
            "expected": expected_seed_contract,
            "actual": run_plan.get("seed_contract"),
        }, sort_keys=True)
    )
legacy_root = Path(run_plan["legacy_root"]).resolve()
overlay_root = Path(run_plan["overlay_root"]).resolve()
python_executable = Path(run_plan["python_executable"]).resolve()
if not python_executable.is_file():
    raise RuntimeError(f"PROGRAM077_PYTHON_MISSING={python_executable}")
if not (overlay_root / "unsloth" / "__init__.py").is_file():
    raise RuntimeError(f"PROGRAM077_UNSLOTH_OVERLAY_MISSING={overlay_root}")
for compiled_name in ["torch", "triton", "numpy"]:
    if (overlay_root / compiled_name).exists():
        raise RuntimeError(f"PROGRAM077_COMPILED_PACKAGE_LEAK={compiled_name}")

allowed_runtime_families = {
    "program050_current": {
        "accelerate": "1.13.0",
        "peft": "0.18.1",
        "bitsandbytes": "0.49.2",
    },
    "program047_legacy": {
        "accelerate": "1.11.0",
        "peft": "0.17.1",
        "bitsandbytes": "0.48.2",
    },
}
runtime_family = run_plan.get("runtime_family")
if runtime_family not in allowed_runtime_families:
    raise RuntimeError(f"PROGRAM077_RUNTIME_FAMILY_INVALID={runtime_family!r}")
expected_pinned_versions = allowed_runtime_families[runtime_family]
selection_attempts = run_plan.get("runtime_selection_attempts")
if not isinstance(selection_attempts, list) or not any(
    item.get("success") is True and item.get("runtime_family") == runtime_family
    for item in selection_attempts
    if isinstance(item, dict)
):
    raise RuntimeError("PROGRAM077_RUNTIME_SELECTION_AUDIT_MISSING")
if run_plan.get("pinned_runtime_versions") != expected_pinned_versions:
    raise RuntimeError(
        "PROGRAM077_RUN_PLAN_RUNTIME_LOCK_MISMATCH="
        + json.dumps({
            "expected": expected_pinned_versions,
            "actual": run_plan.get("pinned_runtime_versions"),
        }, sort_keys=True)
    )
if run_plan.get("pinned_runtime_imported_versions") != expected_pinned_versions:
    raise RuntimeError(
        "PROGRAM077_IMPORTED_RUNTIME_LOCK_MISMATCH="
        + json.dumps(run_plan.get("pinned_runtime_imported_versions"), sort_keys=True)
    )
for report_key in ["pinned_runtime_source", "pinned_runtime_overlay"]:
    reports = run_plan.get(report_key, {})
    for module_name, expected_version in expected_pinned_versions.items():
        report = reports.get(module_name, {})
        if report.get("version") != expected_version or not report.get("record_tree_sha256"):
            raise RuntimeError(
                "PROGRAM077_RUNTIME_FINGERPRINT_MISSING="
                + json.dumps({"report_key": report_key, "module": module_name, "report": report}, sort_keys=True)
            )
        if report.get("record_tree_sha256") != run_plan["pinned_runtime_source"][module_name]["record_tree_sha256"]:
            raise RuntimeError(
                "PROGRAM077_RUNTIME_FINGERPRINT_MISMATCH="
                + json.dumps({"report_key": report_key, "module": module_name}, sort_keys=True)
            )
print("PROGRAM077_RUNTIME_LOCK_RECHECK_OK=" + json.dumps({
    "runtime_family": runtime_family,
    "versions": expected_pinned_versions,
    "tree_sha256": {
        module_name: run_plan["pinned_runtime_source"][module_name]["record_tree_sha256"]
        for module_name in sorted(expected_pinned_versions)
    },
}, sort_keys=True))

qwen3_probe = run_plan.get("qwen3_official_kv_probe", {})
if qwen3_probe.get("backend") != "official_style_kv_cache":
    raise RuntimeError(f"PROGRAM077_QWEN3_PROBE_INVALID={qwen3_probe}")

run_env = os.environ.copy()
run_env.update({
    "PROGRAM077_LEGACY_ROOT": str(legacy_root),
    "PROGRAM077_OVERLAY_ROOT": str(overlay_root),
    "PYTHONPATH": run_plan["runtime_pythonpath"],
    "UNSLOTH_DISABLE_STATISTICS": "1",
    "TRITON_PTXAS_PATH": run_plan["ptxas_path"],
    "OMP_NUM_THREADS": "12",
    "PYTHONHASHSEED": run_plan["seed_contract"]["python_hash_seed"],
    "PROGRAM077_BASE_SEED": str(run_plan["seed_contract"]["base_seed"]),
    "PROGRAM077_SEED_POLICY": "rng_only_no_backend_change",
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
})
subprocess.run(
    [str(python_executable), "-m", "py_compile", "arc_loader.py", "arc_decoder.py", "arc_solver.py", "starter.py"],
    env=run_env,
    cwd="/kaggle/working",
    check=True,
)

# A one-step production-runtime smoke test catches binary/API failures before the four workers load models.
bootstrap = '''import os\nimport sys\nfrom pathlib import Path\nlegacy_root = Path(os.environ["PROGRAM077_LEGACY_ROOT"]).resolve()\noverlay_root = Path(os.environ["PROGRAM077_OVERLAY_ROOT"]).resolve()\nclean = []\nfor item in sys.path:\n    if not item:\n        clean.append(item)\n        continue\n    try:\n        resolved = Path(item).resolve()\n    except OSError:\n        continue\n    if resolved == legacy_root or legacy_root in resolved.parents or resolved == overlay_root:\n        continue\n    clean.append(item)\nsys.path[:] = [str(overlay_root)] + clean\n'''
smoke_path = Path("/kaggle/working/program077_model_smoke.py")
smoke_path.write_text(
    bootstrap
    + r'''
import hashlib
import json
import math
import os
import pickle
import random

import numpy as np
import torch
import torch.nn.functional as F

BASE_SEED = int(os.environ["PROGRAM077_BASE_SEED"])
if os.environ.get("PYTHONHASHSEED") != "0":
    raise RuntimeError(f"PROGRAM077_SMOKE_PYTHONHASHSEED={os.environ.get('PYTHONHASHSEED')!r}")
random.seed(BASE_SEED)
np.random.seed(BASE_SEED)
torch.manual_seed(BASE_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(BASE_SEED)
print("PROGRAM077_SMOKE_SEED_READY=" + json.dumps({
    "base_seed": BASE_SEED,
    "python_hash_seed": os.environ.get("PYTHONHASHSEED"),
    "hash_probe": hash("arc-program-seed-probe-v1"),
    "python_rng_sha256": hashlib.sha256(pickle.dumps(random.getstate(), protocol=4)).hexdigest(),
    "numpy_rng_sha256": hashlib.sha256(pickle.dumps(np.random.get_state(), protocol=4)).hexdigest(),
    "torch_cpu_rng_sha256": hashlib.sha256(torch.get_rng_state().cpu().numpy().tobytes()).hexdigest(),
    "deterministic_algorithms_enabled": bool(torch.are_deterministic_algorithms_enabled()),
    "cudnn_deterministic": bool(torch.backends.cudnn.deterministic),
    "cudnn_benchmark": bool(torch.backends.cudnn.benchmark),
}, sort_keys=True))

from unsloth import FastLanguageModel, UnslothTrainingArguments
from peft import get_peft_model_state_dict
from datasets import Dataset
from arc_loader import QwenFormatter
from arc_solver import ARC_TOKENS, QwenDataCollatorForCompletionOnlyLM, UnslothFixedTrainer

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
    full_finetuning=False,
    load_in_4bit=False,
    local_files_only=True,
    use_gradient_checkpointing=False,
    max_seq_length=8192,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=256,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing=False,
    random_state=42,
    use_rslora=True,
    loftq_config=None,
)
for _, parameter in model.named_parameters():
    if parameter.dtype == torch.float32:
        parameter.data = parameter.data.to(torch.bfloat16)
state = get_peft_model_state_dict(model, adapter_name="default")
if not state:
    raise RuntimeError("PROGRAM077_LORA_STATE_EMPTY")

formatter = QwenFormatter(tokenizer)
smoke_text = formatter.fmt_train([
    {"input": [[1, 0], [0, 1]], "output": [[1, 0], [0, 1]]},
    {"input": [[2, 0], [0, 2]], "output": [[2, 0], [0, 2]]},
])
collator = QwenDataCollatorForCompletionOnlyLM(tokenizer=tokenizer, mlm=False)
args = UnslothTrainingArguments(
    output_dir="/kaggle/working/program077_smoke_output",
    per_device_eval_batch_size=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    num_train_epochs=1,
    max_steps=1,
    warmup_steps=0,
    warmup_ratio=0.1,
    max_grad_norm=1.0,
    learning_rate=5e-5,
    optim="adamw_torch",
    weight_decay=0.0,
    lr_scheduler_type="cosine",
    seed=42,
    report_to="none",
    save_strategy="no",
    eval_strategy="no",
    logging_strategy="no",
    fp16=False,
    bf16=True,
    fsdp="",
    ddp_find_unused_parameters=False,
    dataloader_num_workers=0,
    gradient_checkpointing=False,
)
model = FastLanguageModel.for_training(model)
trainer = UnslothFixedTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=collator,
    train_dataset=Dataset.from_list([{"text": smoke_text}]),
    dataset_text_field="text",
    max_seq_length=8192,
    args=args,
)
trainer.create_optimizer_and_scheduler(num_training_steps=1)
batch = next(iter(trainer.get_train_dataloader()))
batch = {
    key: (value.to(model.device) if torch.is_tensor(value) else value)
    for key, value in batch.items()
}
if "labels" not in batch or not torch.any(batch["labels"] != -100):
    raise RuntimeError("PROGRAM077_SMOKE_NO_SUPERVISED_LABELS")
loss = trainer.compute_loss(model, batch)
if loss is None or not math.isfinite(float(loss.detach().cpu())):
    raise RuntimeError(f"PROGRAM077_SMOKE_INVALID_LOSS={loss}")
loss.backward()
torch.cuda.synchronize()
grad_tensors = sum(
    1 for parameter in model.parameters()
    if parameter.requires_grad and parameter.grad is not None
)
if grad_tensors == 0:
    raise RuntimeError("PROGRAM077_SMOKE_NO_GRADIENTS")
trainer.optimizer.step()
trainer.optimizer.zero_grad(set_to_none=True)
model = FastLanguageModel.for_inference(model)
def cache_full_parity(ids, label):
    prefix_mask = torch.ones_like(ids)
    prefix_output_local = model(
        input_ids=ids,
        attention_mask=prefix_mask,
        return_dict=True,
        use_cache=True,
    )
    if prefix_output_local.past_key_values is None or not torch.isfinite(prefix_output_local.logits).all():
        raise RuntimeError(f"PROGRAM077_SMOKE_PREFIX_INFERENCE_FAILED label={label}")
    next_input_local = ids[:, -1:]
    next_position_local = torch.full(
        (ids.shape[0], 1),
        ids.shape[1],
        dtype=torch.long,
        device=ids.device,
    )
    cached_output = model(
        input_ids=next_input_local,
        position_ids=next_position_local,
        past_key_values=prefix_output_local.past_key_values,
        return_dict=True,
        use_cache=True,
    )
    full_ids = torch.cat([ids, next_input_local], dim=1)
    full_output = model(
        input_ids=full_ids,
        attention_mask=torch.ones_like(full_ids),
        return_dict=True,
        use_cache=False,
    )
    cached_logits = cached_output.logits[:, -1, :].float()
    full_logits = full_output.logits[:, -1, :].float()
    if cached_logits.shape != full_logits.shape:
        raise RuntimeError(
            f"PROGRAM077_SMOKE_CACHE_FULL_SHAPE label={label} cached={cached_logits.shape} full={full_logits.shape}"
        )
    if not torch.isfinite(cached_logits).all() or not torch.isfinite(full_logits).all():
        raise RuntimeError(f"PROGRAM077_SMOKE_CACHE_FULL_NONFINITE label={label}")
    delta = cached_logits - full_logits
    absolute_delta = delta.abs()
    max_abs = float(absolute_delta.max().cpu())
    mean_abs = float(absolute_delta.mean().cpu())
    rmse = float(torch.sqrt(torch.mean(delta.square())).cpu())
    reference_rms = float(torch.sqrt(torch.mean(full_logits.square())).cpu())
    relative_rmse = rmse / max(reference_rms, 1e-6)
    cosine = F.cosine_similarity(cached_logits, full_logits, dim=-1)
    cosine_min = float(cosine.min().cpu())
    top1_agreement = float(
        (cached_logits.argmax(dim=-1) == full_logits.argmax(dim=-1)).float().mean().cpu()
    )
    arc_token_ids = torch.as_tensor(ARC_TOKENS, dtype=torch.long, device=cached_logits.device)
    if int(arc_token_ids.max().item()) >= cached_logits.shape[-1]:
        raise RuntimeError(
            f"PROGRAM077_SMOKE_ARC_TOKEN_RANGE max_token={int(arc_token_ids.max().item())} "
            f"vocab={cached_logits.shape[-1]}"
        )
    cached_arc_nll = (
        torch.logsumexp(cached_logits, dim=-1, keepdim=True)
        - cached_logits.index_select(-1, arc_token_ids)
    )
    full_arc_nll = (
        torch.logsumexp(full_logits, dim=-1, keepdim=True)
        - full_logits.index_select(-1, arc_token_ids)
    )
    arc_nll_delta = (cached_arc_nll - full_arc_nll).abs()
    dfs_max_score = -math.log(0.2)
    cached_alive = cached_arc_nll < dfs_max_score
    full_alive = full_arc_nll < dfs_max_score
    threshold_mismatch_count = int((cached_alive != full_alive).sum().item())
    threshold_decision_count = int(cached_alive.numel())
    threshold_decision_agreement = 1.0 - (
        threshold_mismatch_count / max(threshold_decision_count, 1)
    )
    arc_nll_max_abs = float(arc_nll_delta.max().cpu())
    arc_nll_mean_abs = float(arc_nll_delta.mean().cpu())
    full_threshold_margin_min = float((full_arc_nll - dfs_max_score).abs().min().cpu())
    top_k = min(4, cached_arc_nll.shape[-1])
    cached_topk = torch.topk(-cached_arc_nll, k=top_k, dim=-1).indices
    full_topk = torch.topk(-full_arc_nll, k=top_k, dim=-1).indices
    topk_position_agreement = float((cached_topk == full_topk).float().mean().cpu())
    # Bfloat16 cached and full kernels need not be bit-identical. Scale-aware
    # error, cosine direction, and the production batch's argmaxes must agree.
    if max_abs > 4.0 or relative_rmse > 0.08 or cosine_min < 0.99 or top1_agreement < 0.75:
        raise RuntimeError(
            "PROGRAM077_SMOKE_CACHE_FULL_PARITY_FAILED="
            + json.dumps({
                "label": label,
                "max_abs": max_abs,
                "mean_abs": mean_abs,
                "rmse": rmse,
                "reference_rms": reference_rms,
                "relative_rmse": relative_rmse,
                "cosine_min": cosine_min,
                "top1_agreement": top1_agreement,
            }, sort_keys=True)
        )
    return prefix_output_local, cached_output, {
        "label": label,
        "max_abs": max_abs,
        "mean_abs": mean_abs,
        "rmse": rmse,
        "reference_rms": reference_rms,
        "relative_rmse": relative_rmse,
        "cosine_min": cosine_min,
        "top1_agreement": top1_agreement,
        "arc_nll_max_abs": arc_nll_max_abs,
        "arc_nll_mean_abs": arc_nll_mean_abs,
        "dfs_threshold": dfs_max_score,
        "threshold_mismatch_count": threshold_mismatch_count,
        "threshold_decision_count": threshold_decision_count,
        "threshold_decision_agreement": threshold_decision_agreement,
        "full_threshold_margin_min": full_threshold_margin_min,
        "arc_top4_position_agreement": topk_position_agreement,
    }

with torch.no_grad():
    base_prefix = batch["input_ids"][:, : min(8, batch["input_ids"].shape[1])]
    prefix_ids = base_prefix.repeat(4, 1)
    prefix_output, next_output, batch4_parity = cache_full_parity(prefix_ids, "batch4")
    _, batch1_output, batch1_parity = cache_full_parity(base_prefix[:1], "batch1")
    if next_output.logits.shape[0] != 4 or next_output.logits.shape[1] != 1:
        raise RuntimeError(f"PROGRAM077_SMOKE_CACHED_SHAPE={next_output.logits.shape}")
    if batch1_output.logits.shape[0] != 1 or batch1_output.logits.shape[1] != 1:
        raise RuntimeError(f"PROGRAM077_SMOKE_BATCH1_CACHED_SHAPE={batch1_output.logits.shape}")
print("PROGRAM077_MODEL_SMOKE_OK=" + json.dumps({
    "loss": float(loss.detach().cpu()),
    "lora_tensors": len(state),
    "gradient_tensors": grad_tensors,
    "batch_tokens": int(batch["input_ids"].numel()),
    "supervised_tokens": int((batch["labels"] != -100).sum().item()),
    "prefix_logits_shape": list(prefix_output.logits.shape),
    "cached_logits_shape": list(next_output.logits.shape),
    "batch1_cached_logits_shape": list(batch1_output.logits.shape),
    "batch4_cache_full_parity": batch4_parity,
    "batch1_cache_full_parity": batch1_parity,
    "allocated_mb": round(torch.cuda.memory_allocated(0) / 1024 / 1024, 1),
    "reserved_mb": round(torch.cuda.memory_reserved(0) / 1024 / 1024, 1),
    "device": torch.cuda.get_device_name(0),
}, sort_keys=True))
''',
    encoding="utf-8",
)
smoke_env = run_env.copy()
smoke_env.update({
    "CUDA_VISIBLE_DEVICES": "0",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
})
smoke = subprocess.run(
    [str(python_executable), str(smoke_path)],
    env=smoke_env,
    cwd="/kaggle/working",
    text=True,
    capture_output=True,
    timeout=600,
)
if smoke.stdout:
    print(smoke.stdout, end="")
if smoke.returncode != 0:
    if smoke.stderr:
        print(smoke.stderr, file=sys.stderr, end="")
    raise RuntimeError(f"PROGRAM077_MODEL_SMOKE_FAILED exit_code={smoke.returncode}")
if "PROGRAM077_SMOKE_SEED_READY=" not in smoke.stdout:
    raise RuntimeError("PROGRAM077_SMOKE_SEED_MARKER_MISSING")
if "PROGRAM077_MODEL_SMOKE_OK=" not in smoke.stdout:
    raise RuntimeError("PROGRAM077_MODEL_SMOKE_MARKER_MISSING")

from collections import Counter
import bz2
import io
import math
import pickle
import shutil
import tarfile


MAX_DEEP_TASKS = 0
DEEP_WALL_LIMIT_SECONDS = 0
MIN_DEEP_WINDOW_SECONDS = 0
PRIMARY_RUNTIME_REFERENCE_SECONDS = 18_700
DEEP_TASK_REFERENCE_SECONDS = 1_500
DEEP_PROCESS_OVERHEAD_SECONDS = 300
PRIMARY_CHECKPOINT_REFERENCE_SECONDS = 300
DEEP_BASE_SEED = None
MIN_PRIMARY_PRESENT_RATIO = 0.94
QUALITY_SCANNER_TIMEOUT_SECONDS = 300


def forecast_runtime_seconds(deep_task_count):
    deep_task_count = max(0, int(deep_task_count))
    waves = math.ceil(deep_task_count / 4) if deep_task_count else 0
    estimate = PRIMARY_RUNTIME_REFERENCE_SECONDS + PRIMARY_CHECKPOINT_REFERENCE_SECONDS + (
        DEEP_PROCESS_OVERHEAD_SECONDS + waves * DEEP_TASK_REFERENCE_SECONDS
        if deep_task_count else 0
    )
    return min(estimate, 10 * 3600)


def expected_basekeys_by_task(challenge):
    return {
        task_id: [f"{task_id}_{test_index}" for test_index in range(len(task["test"]))]
        for task_id, task in sorted(challenge.items())
    }


def grid_fingerprint(solution):
    # Fail-safe fallback only. Do not import NumPy in the notebook kernel: the
    # selected legacy asset contains an intentionally incomplete NumPy tree.
    # The authoritative isolated scanner below imports NumPy only after its
    # bootstrap has removed that legacy path.
    rows = solution.tolist() if hasattr(solution, "tolist") else solution
    if not isinstance(rows, (list, tuple)) or not 1 <= len(rows) <= 30:
        raise ValueError("solution is not a valid row sequence")
    width = None
    result = []
    for row in rows:
        if not isinstance(row, (list, tuple)):
            raise ValueError("solution row is not a sequence")
        if width is None:
            width = len(row)
        if not 1 <= len(row) <= 30 or len(row) != width:
            raise ValueError("solution is not a rectangular 1..30 grid")
        converted = []
        for value in row:
            if isinstance(value, (str, bytes)):
                raise ValueError("grid contains a non-numeric value")
            integer = int(value)
            if value != integer:
                raise ValueError("grid contains a non-integer value")
            if not 0 <= integer <= 9:
                raise ValueError("grid values must be in 0..9")
            converted.append(integer)
        result.append(tuple(converted))
    return tuple(result)


def snapshot_output_quality(output_dir, expected_by_task):
    output_dir = Path(output_dir)
    files = sorted(path for path in output_dir.iterdir() if path.is_file()) if output_dir.is_dir() else []
    task_file_counts = Counter(path.name.split("_", 1)[0] for path in files)
    fingerprints = {
        basekey: set()
        for basekeys in expected_by_task.values()
        for basekey in basekeys
    }
    valid_sample_counts = Counter()
    corrupt_files = []
    for path in files:
        basekey = path.name.split(".", 1)[0]
        if basekey not in fingerprints:
            continue
        try:
            with bz2.BZ2File(path) as handle:
                outputs = pickle.load(handle)
            if not isinstance(outputs, list):
                raise TypeError(f"decoded payload must be a list, got {type(outputs).__name__}")
            temporary = []
            for sample in outputs:
                if not isinstance(sample, dict) or not {"solution", "beam_score", "score_aug"}.issubset(sample):
                    continue
                temporary.append(grid_fingerprint(sample["solution"]))
            for fingerprint in temporary:
                fingerprints[basekey].add(fingerprint)
                valid_sample_counts[basekey] += 1
        except Exception as exc:
            corrupt_files.append({
                "name": path.name,
                "error": f"{type(exc).__name__}: {exc}",
            })

    unique_candidate_counts = {
        basekey: len(values)
        for basekey, values in sorted(fingerprints.items())
    }
    present_basekeys = sorted(
        basekey for basekey, count in unique_candidate_counts.items() if count > 0
    )
    missing_basekeys = sorted(
        basekey for basekey, count in unique_candidate_counts.items() if count == 0
    )
    starved_basekeys = sorted(
        basekey for basekey, count in unique_candidate_counts.items() if count < 2
    )
    missing_by_task = {}
    starved_by_task = {}
    for task_id, basekeys in expected_by_task.items():
        missing = [basekey for basekey in basekeys if unique_candidate_counts[basekey] == 0]
        starved = [basekey for basekey in basekeys if unique_candidate_counts[basekey] < 2]
        if missing:
            missing_by_task[task_id] = missing
        if starved:
            starved_by_task[task_id] = starved
    return {
        "file_count": len(files),
        "present_basekeys": present_basekeys,
        "missing_basekeys": missing_basekeys,
        "starved_basekeys": starved_basekeys,
        "missing_by_task": dict(sorted(missing_by_task.items())),
        "starved_by_task": dict(sorted(starved_by_task.items())),
        "unique_candidate_counts": unique_candidate_counts,
        "valid_sample_counts": dict(sorted(valid_sample_counts.items())),
        "task_file_counts": dict(sorted(task_file_counts.items())),
        "corrupt_files": corrupt_files,
    }


def snapshot_output_quality_isolated(
    output_dir,
    expected_by_task,
    scan_label,
    scanner_env,
    working_dir=Path("/kaggle/working"),
):
    """Unpickle solver outputs only inside the verified production runtime."""
    working_dir = Path(working_dir)
    working_dir.mkdir(parents=True, exist_ok=True)
    scanner_path = working_dir / "program077_quality_scanner.py"
    request_path = working_dir / f"program077_quality_request_{scan_label}.json"
    result_path = working_dir / f"program077_quality_result_{scan_label}.json"
    request_path.write_text(
        json.dumps({
            "output_dir": str(Path(output_dir)),
            "expected_by_task": expected_by_task,
            "result_path": str(result_path),
        }, sort_keys=True),
        encoding="utf-8",
    )
    scanner_body = r'''
import bz2
import json
import pickle
import sys
from collections import Counter
from pathlib import Path

import numpy as np


def grid_fingerprint(solution):
    array = np.asarray(solution)
    if array.ndim != 2 or not 1 <= array.shape[0] <= 30 or not 1 <= array.shape[1] <= 30:
        raise ValueError(f"invalid grid shape {array.shape}")
    if not np.issubdtype(array.dtype, np.integer):
        if not np.all(np.equal(array, np.floor(array))):
            raise ValueError("grid contains non-integer values")
    array = array.astype(int)
    if np.any(array < 0) or np.any(array > 9):
        raise ValueError("grid values must be in 0..9")
    return tuple(tuple(int(value) for value in row) for row in array)


request = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
output_dir = Path(request["output_dir"])
expected_by_task = request["expected_by_task"]
files = sorted(path for path in output_dir.iterdir() if path.is_file()) if output_dir.is_dir() else []
task_file_counts = Counter(path.name.split("_", 1)[0] for path in files)
fingerprints = {
    basekey: set()
    for basekeys in expected_by_task.values()
    for basekey in basekeys
}
valid_sample_counts = Counter()
corrupt_files = []
for path in files:
    basekey = path.name.split(".", 1)[0]
    if basekey not in fingerprints:
        continue
    try:
        with bz2.BZ2File(path) as handle:
            outputs = pickle.load(handle)
        if not isinstance(outputs, list):
            raise TypeError(f"decoded payload must be a list, got {type(outputs).__name__}")
        temporary = []
        for sample in outputs:
            if not isinstance(sample, dict) or not {"solution", "beam_score", "score_aug"}.issubset(sample):
                continue
            temporary.append(grid_fingerprint(sample["solution"]))
        for fingerprint in temporary:
            fingerprints[basekey].add(fingerprint)
            valid_sample_counts[basekey] += 1
    except Exception as exc:
        corrupt_files.append({
            "name": path.name,
            "error": f"{type(exc).__name__}: {exc}",
        })

unique_candidate_counts = {
    basekey: len(values)
    for basekey, values in sorted(fingerprints.items())
}
present_basekeys = sorted(basekey for basekey, count in unique_candidate_counts.items() if count > 0)
missing_basekeys = sorted(basekey for basekey, count in unique_candidate_counts.items() if count == 0)
starved_basekeys = sorted(basekey for basekey, count in unique_candidate_counts.items() if count < 2)
missing_by_task = {}
starved_by_task = {}
for task_id, basekeys in expected_by_task.items():
    missing = [basekey for basekey in basekeys if unique_candidate_counts[basekey] == 0]
    starved = [basekey for basekey in basekeys if unique_candidate_counts[basekey] < 2]
    if missing:
        missing_by_task[task_id] = missing
    if starved:
        starved_by_task[task_id] = starved
report = {
    "scanner_ok": True,
    "scanner_python": sys.executable,
    "file_count": len(files),
    "present_basekeys": present_basekeys,
    "missing_basekeys": missing_basekeys,
    "starved_basekeys": starved_basekeys,
    "missing_by_task": dict(sorted(missing_by_task.items())),
    "starved_by_task": dict(sorted(starved_by_task.items())),
    "unique_candidate_counts": unique_candidate_counts,
    "valid_sample_counts": dict(sorted(valid_sample_counts.items())),
    "task_file_counts": dict(sorted(task_file_counts.items())),
    "corrupt_files": corrupt_files,
}
Path(request["result_path"]).write_text(
    json.dumps(report, sort_keys=True, separators=(",", ":")),
    encoding="utf-8",
)
print("PROGRAM077_QUALITY_SCANNER_OK=" + json.dumps({
    "file_count": len(files),
    "present_basekey_count": len(present_basekeys),
    "missing_basekey_count": len(missing_basekeys),
    "starved_basekey_count": len(starved_basekeys),
    "corrupt_file_count": len(corrupt_files),
}, sort_keys=True))
'''
    scanner_path.write_text(bootstrap + scanner_body.lstrip("\n"), encoding="utf-8")
    result_path.unlink(missing_ok=True)
    try:
        scan = subprocess.run(
            [str(python_executable), str(scanner_path), str(request_path)],
            env=scanner_env,
            cwd=str(working_dir),
            text=True,
            capture_output=True,
            timeout=QUALITY_SCANNER_TIMEOUT_SECONDS,
        )
        if scan.stdout:
            print(scan.stdout, end="")
        if scan.returncode != 0 or not result_path.is_file():
            raise RuntimeError(
                f"scanner_return_code={scan.returncode} stderr={scan.stderr[-2000:]}"
            )
        report = json.loads(result_path.read_text(encoding="utf-8"))
        if report.get("scanner_ok") is not True:
            raise RuntimeError("scanner result did not assert scanner_ok")
        return report
    except Exception as exc:
        # Preserve the primary pass and let the early gate BLOCK safely. The
        # finalizer independently recomputes eligibility in this same runtime.
        fallback = snapshot_output_quality(output_dir, expected_by_task)
        fallback["scanner_ok"] = False
        fallback["scanner_error"] = f"{type(exc).__name__}: {exc}"
        print("PROGRAM077_QUALITY_SCANNER_FAILED_RECOVERABLE=" + json.dumps({
            "scan_label": scan_label,
            "error": fallback["scanner_error"],
        }, sort_keys=True))
        return fallback


def run_candidate_pickle_fixture_gate(
    scanner_env,
    working_dir=Path("/kaggle/working"),
):
    """Prove candidate production and isolated decoding before the long primary pass."""
    working_dir = Path(working_dir)
    fixture_dir = working_dir / "program077_preflight_candidate_fixture"
    producer_path = working_dir / "program077_candidate_fixture_producer.py"
    if fixture_dir.exists():
        shutil.rmtree(fixture_dir)
    fixture_dir.mkdir(parents=True)
    producer_body = r'''
import bz2
import pickle
import sys
from pathlib import Path

import numpy as np


root = Path(sys.argv[1])
root.mkdir(parents=True, exist_ok=True)
payloads = {
    "fixture0_0.preflight": [
        {"solution": np.asarray([[1, 0], [0, 1]], dtype=int), "beam_score": 0.1, "score_aug": [0.2] * 8},
        {"solution": np.asarray([[1, 1], [0, 1]], dtype=int), "beam_score": 0.2, "score_aug": [0.3] * 8},
    ],
    "fixture0_1.preflight": [
        {"solution": np.asarray([[2]], dtype=int), "beam_score": 0.3, "score_aug": [0.4] * 8},
    ],
}
for name, samples in payloads.items():
    with bz2.BZ2File(root / name, "w") as handle:
        pickle.dump(samples, handle)
print("PROGRAM077_CANDIDATE_PICKLE_FIXTURE_PRODUCED=2")
'''
    producer_path.write_text(bootstrap + producer_body.lstrip("\n"), encoding="utf-8")
    try:
        produced = subprocess.run(
            [str(python_executable), str(producer_path), str(fixture_dir)],
            env=scanner_env,
            cwd=str(working_dir),
            text=True,
            capture_output=True,
            timeout=120,
        )
        if produced.stdout:
            print(produced.stdout, end="")
        if produced.returncode != 0:
            raise RuntimeError(
                f"fixture_producer_return_code={produced.returncode} stderr={produced.stderr[-2000:]}"
            )
        report = snapshot_output_quality_isolated(
            fixture_dir,
            {"fixture0": ["fixture0_0", "fixture0_1"]},
            "preflight_fixture",
            scanner_env,
            working_dir=working_dir,
        )
        expected_counts = {"fixture0_0": 2, "fixture0_1": 1}
        errors = []
        if report.get("scanner_ok") is not True:
            errors.append("scanner_not_ok")
        if report.get("file_count") != 2:
            errors.append("file_count")
        if report.get("present_basekeys") != ["fixture0_0", "fixture0_1"]:
            errors.append("present_basekeys")
        if report.get("missing_basekeys") != []:
            errors.append("missing_basekeys")
        if report.get("starved_basekeys") != ["fixture0_1"]:
            errors.append("starved_basekeys")
        if report.get("unique_candidate_counts") != expected_counts:
            errors.append("unique_candidate_counts")
        if report.get("corrupt_files") != []:
            errors.append("corrupt_files")
        fixture_report = {
            "ok": not errors,
            "errors": errors,
            "scanner_python": report.get("scanner_python"),
            "observed": report,
        }
        print("PROGRAM077_CANDIDATE_PICKLE_FIXTURE_REPORT=" + json.dumps(fixture_report, sort_keys=True))
        if errors:
            print("PROGRAM077_CANDIDATE_PICKLE_FIXTURE_GATE=FAILED")
            raise RuntimeError("PROGRAM077_CANDIDATE_PICKLE_FIXTURE_GATE_FAILED")
        print("PROGRAM077_CANDIDATE_PICKLE_FIXTURE_GATE=OK")
        return fixture_report
    finally:
        if fixture_dir.exists():
            shutil.rmtree(fixture_dir)


def _sha256_path(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_primary_checkpoint(archive_path, core_manifest):
    archive_path = Path(archive_path)
    expected_files = {entry["archive_name"]: entry for entry in core_manifest["files"]}
    expected_names = ["manifest.json"] + sorted(expected_files)
    with tarfile.open(archive_path, "r") as archive:
        members = archive.getmembers()
        names = [member.name for member in members]
        if names != expected_names:
            raise RuntimeError(f"checkpoint member mismatch: {names[:5]}")
        if any(not member.isfile() for member in members):
            raise RuntimeError("checkpoint contains a non-regular member")
        manifest_handle = archive.extractfile("manifest.json")
        if manifest_handle is None:
            raise RuntimeError("checkpoint manifest is unreadable")
        archived_manifest = json.loads(manifest_handle.read().decode("utf-8"))
        if archived_manifest != core_manifest:
            raise RuntimeError("checkpoint embedded manifest mismatch")
        for name, expected in sorted(expected_files.items()):
            member = archive.getmember(name)
            if member.size != expected["size"]:
                raise RuntimeError(f"checkpoint size mismatch: {name}")
            handle = archive.extractfile(member)
            if handle is None:
                raise RuntimeError(f"checkpoint member unreadable: {name}")
            digest = hashlib.sha256()
            for chunk in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(chunk)
            if digest.hexdigest() != expected["sha256"]:
                raise RuntimeError(f"checkpoint hash mismatch: {name}")
    return {"ok": True, "verified_member_count": len(expected_names)}


def create_verified_primary_checkpoint(
    primary_output_dir,
    challenge_path,
    source_hashes,
    checkpoint_root=Path("/kaggle/working"),
):
    checkpoint_root = Path(checkpoint_root)
    checkpoint_root.mkdir(parents=True, exist_ok=True)
    archive_path = checkpoint_root / "program077_primary_checkpoint.tar"
    manifest_path = checkpoint_root / "program077_primary_checkpoint_manifest.json"
    temporary_path = checkpoint_root / "program077_primary_checkpoint.tmp.tar"
    temporary_path.unlink(missing_ok=True)
    archive_path.unlink(missing_ok=True)
    manifest_path.unlink(missing_ok=True)
    try:
        files = sorted(path for path in Path(primary_output_dir).iterdir() if path.is_file())
        names = [path.name for path in files]
        if len(names) != len(set(names)) or any(Path(name).name != name for name in names):
            raise RuntimeError("unsafe or duplicate primary checkpoint filename")
        entries = [
            {
                "name": path.name,
                "archive_name": "outputs/" + path.name,
                "size": path.stat().st_size,
                "sha256": _sha256_path(path),
            }
            for path in files
        ]
        core_manifest = {
            "program_no": PROGRAM_NO,
            "challenge_sha256": _sha256_path(challenge_path),
            "source_sha256": dict(sorted(source_hashes.items())),
            "file_count": len(entries),
            "files": entries,
        }
        manifest_bytes = json.dumps(
            core_manifest, sort_keys=True, separators=(",", ":"), allow_nan=False
        ).encode("utf-8")
        with tarfile.open(temporary_path, "w", format=tarfile.GNU_FORMAT) as archive:
            manifest_info = tarfile.TarInfo("manifest.json")
            manifest_info.size = len(manifest_bytes)
            manifest_info.mode = 0o644
            manifest_info.uid = manifest_info.gid = 0
            manifest_info.uname = manifest_info.gname = ""
            manifest_info.mtime = 0
            archive.addfile(manifest_info, io.BytesIO(manifest_bytes))
            for path, entry in zip(files, entries):
                info = tarfile.TarInfo(entry["archive_name"])
                info.size = entry["size"]
                info.mode = 0o644
                info.uid = info.gid = 0
                info.uname = info.gname = ""
                info.mtime = 0
                with path.open("rb") as handle:
                    archive.addfile(info, handle)
        verification = verify_primary_checkpoint(temporary_path, core_manifest)
        temporary_path.replace(archive_path)
        report = {
            "ok": True,
            "archive_path": str(archive_path),
            "manifest_path": str(manifest_path),
            "archive_sha256": _sha256_path(archive_path),
            "archive_size": archive_path.stat().st_size,
            "file_count": len(entries),
            "verification": verification,
        }
        manifest_path.write_text(
            json.dumps(
                {**report, "core_manifest": core_manifest},
                sort_keys=True,
                separators=(",", ":"),
                allow_nan=False,
            ),
            encoding="utf-8",
        )
        return report
    except Exception as exc:
        temporary_path.unlink(missing_ok=True)
        return {
            "ok": False,
            "archive_path": str(archive_path),
            "manifest_path": str(manifest_path),
            "error": f"{type(exc).__name__}: {exc}",
        }


def select_deep_tasks(quality, max_deep_tasks=MAX_DEEP_TASKS):
    missing_by_task = quality["missing_by_task"]
    starved_by_task = quality["starved_by_task"]
    task_file_counts = quality["task_file_counts"]
    ranked = sorted(
        starved_by_task,
        key=lambda task_id: (
            -len(missing_by_task.get(task_id, [])),
            -len(starved_by_task.get(task_id, [])),
            int(task_file_counts.get(task_id, 0)),
            task_id,
        ),
    )
    return ranked[:max_deep_tasks]


def parse_json_markers(text, marker):
    reports = []
    prefix = marker + "="
    for line in text.splitlines():
        if line.startswith(prefix):
            reports.append(json.loads(line[len(prefix):]))
    return reports


def run_early_format_reload_selftest(challenge):
    skeleton = {
        task_id: [
            {"attempt_1": [[0]], "attempt_2": [[0]]}
            for _ in task.get("test", [])
        ]
        for task_id, task in sorted(challenge.items())
    }
    errors = []
    if set(skeleton) != set(challenge):
        errors.append("task_id_mismatch")
    checked_tests = 0
    for task_id, task in sorted(challenge.items()):
        expected_count = len(task.get("test", []))
        if len(skeleton[task_id]) != expected_count:
            errors.append(f"test_count:{task_id}")
            continue
        for item in skeleton[task_id]:
            checked_tests += 1
            if set(item) != {"attempt_1", "attempt_2"}:
                errors.append(f"attempt_keys:{task_id}")
            for grid in item.values():
                if grid != [[0]]:
                    errors.append(f"skeleton_grid:{task_id}")
    path = Path("/kaggle/working/program077_early_format_selftest.json")
    path.write_text(
        json.dumps(skeleton, sort_keys=True, separators=(",", ":"), allow_nan=False),
        encoding="utf-8",
    )
    reloaded = json.loads(path.read_text(encoding="utf-8"))
    if reloaded != skeleton:
        errors.append("post_write_reload_mismatch")
    report = {
        "ok": not errors,
        "task_count": len(challenge),
        "test_count": checked_tests,
        "attempt_count": checked_tests * 2,
        "errors": errors,
        "path": str(path),
    }
    if errors:
        print("PROGRAM077_EARLY_FORMAT_RELOAD_SELFTEST_GATE=FAILED")
        print("PROGRAM077_EARLY_FORMAT_RELOAD_SELFTEST_REPORT=" + json.dumps(report, sort_keys=True))
        raise RuntimeError("PROGRAM077_EARLY_FORMAT_RELOAD_SELFTEST_FAILED")
    print("PROGRAM077_EARLY_FORMAT_RELOAD_SELFTEST_GATE=OK")
    print("PROGRAM077_EARLY_FORMAT_RELOAD_SELFTEST_REPORT=" + json.dumps(report, sort_keys=True))
    return report


def remove_stale_lock(lock_path):
    lock_path = Path(lock_path)
    if not lock_path.exists():
        return
    stale_owner = lock_path.read_text(encoding="utf-8", errors="replace")
    match = re.search(r"pid=(\d+)", stale_owner)
    owner_alive = False
    if match:
        try:
            os.kill(int(match.group(1)), 0)
            owner_alive = True
        except (ProcessLookupError, PermissionError, ValueError):
            owner_alive = False
    if owner_alive:
        raise RuntimeError(f"PROGRAM077_EXISTING_STARTER_ALIVE={stale_owner.strip()}")
    lock_path.unlink()


def run_starter_pass(command, run_label, log_path, pass_env, append):
    remove_stale_lock("/kaggle/working/program077_starter.lock")
    started = time.time()
    mode = "a" if append else "w"
    print(f"Launching Program No. 077 {run_label} pass:", " ".join(command))
    with Path(log_path).open(mode, encoding="utf-8", buffering=1) as starter_log_handle:
        boundary = "PROGRAM077_ORCHESTRATOR_PASS_BEGIN=" + json.dumps({
            "run_label": run_label,
            "started": started,
            "command": command,
        }, sort_keys=True) + "\n"
        starter_log_handle.write(boundary)
        print(boundary, end="", flush=True)
        process = subprocess.Popen(
            command,
            env=pass_env,
            cwd="/kaggle/working",
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        if process.stdout is None:
            raise RuntimeError(f"PROGRAM077_STARTER_PIPE_MISSING run_label={run_label}")
        for line in process.stdout:
            starter_log_handle.write(line)
            print(line, end="", flush=True)
        return_code = process.wait()
        ended = time.time()
        boundary = "PROGRAM077_ORCHESTRATOR_PASS_END=" + json.dumps({
            "run_label": run_label,
            "ended": ended,
            "elapsed_seconds": round(ended - started, 3),
            "return_code": return_code,
        }, sort_keys=True) + "\n"
        starter_log_handle.write(boundary)
        print(boundary, end="", flush=True)
    return {
        "run_label": run_label,
        "started": started,
        "ended": ended,
        "elapsed_seconds": round(ended - started, 3),
        "return_code": return_code,
    }


print(
    "PROGRAM077_CODE_PREFLIGHT_OK "
    f"profile={run_plan['runtime_profile']} python={python_executable} "
    f"runtime_family={runtime_family} runtime_lock=pinned qwen3_attention={qwen3_probe['backend']} "
    f"exact_solver_sha256={actual_hashes['arc_solver.py']} "
    f"backend_policy=normal_nondeterministic_cuda overlay={overlay_root}"
)
print("PROGRAM077_RUNTIME_FORECAST=" + json.dumps({
    "hardware": "NVIDIA_L4_X4",
    "primary_reference_seconds": PRIMARY_RUNTIME_REFERENCE_SECONDS,
    "primary_checkpoint_reference_seconds": PRIMARY_CHECKPOINT_REFERENCE_SECONDS,
    "deep_task_count": 0,
    "forecast_total_seconds": forecast_runtime_seconds(0),
    "typical_range_text": "5h10m-9h15m_primary_plus_15m_finalization",
    "central_estimate_text": "about_5h40m_visible_hidden_may_use_full_deadline",
    "hard_notebook_limit_seconds": 9 * 3600 + 30 * 60,
    "primary_work_deadline_seconds": 9 * 3600 + 15 * 60,
    "deep_wall_limit_seconds": 0,
}, sort_keys=True))

# This inexpensive write/reload check runs before the multi-hour primary pass.
# It catches path, JSON serialization, task-count, and attempt-key failures early.
early_format_selftest = run_early_format_reload_selftest(challenge_scope_payload)
# This exercises the exact producer -> BZ2/pickle -> isolated scanner contract.
# Any environment incompatibility stops here, before the multi-hour primary pass.
candidate_pickle_fixture = run_candidate_pickle_fixture_gate(run_env)

starter_log_path = Path("/kaggle/working/program077_starter.log")
primary_output_dir = Path("/kaggle/inference_outputs_primary")
deep_output_dir = Path("/kaggle/inference_outputs_deep")
default_output_dir = Path("/kaggle/inference_outputs")
for stale_dir in (primary_output_dir, deep_output_dir, default_output_dir):
    if stale_dir.exists():
        shutil.rmtree(stale_dir)

primary_command = [
    str(python_executable),
    "starter.py",
    "--end-time",
    str(run_plan["global_end_time"]),
    "--challenge-path",
    run_plan["challenge_path"],
    "--run-label",
    "primary",
]
primary_pass = run_starter_pass(
    primary_command, "primary", starter_log_path, run_env, append=False
)
primary_status = "COMPLETED" if primary_pass["return_code"] == 0 else "FAILED_PARTIAL_RECOVERABLE"
if not default_output_dir.is_dir():
    default_output_dir.mkdir(parents=True, exist_ok=True)

expected_by_task = expected_basekeys_by_task(challenge_scope_payload)
primary_quality = snapshot_output_quality_isolated(
    default_output_dir, expected_by_task, "primary", run_env
)
deep_tasks = []  # Program077 Primary-only policy
deep_task_file = Path("/kaggle/working/program077_deep_tasks.json")
deep_task_file.write_text(json.dumps(deep_tasks, indent=2), encoding="utf-8")

# All facts needed to reject a broken primary pass are available now. Audit
# them before launching the second multi-hour GPU pass.
primary_log_text_early = starter_log_path.read_text(encoding="utf-8", errors="replace")
primary_worker_reports_early = [
    report for report in parse_json_markers(primary_log_text_early, "PROGRAM077_WORKER_SEED_READY")
    if report.get("run_label") == "primary"
]
primary_task_reports_early = [
    report for report in parse_json_markers(primary_log_text_early, "PROGRAM077_TASK_SEED_READY")
    if report.get("run_label") == "primary"
]
primary_task_counter_early = Counter(report.get("task_id") for report in primary_task_reports_early)
primary_worker_seed_ok_early = sorted(
    report.get("rank") for report in primary_worker_reports_early
) == [0, 1, 2, 3]
primary_task_coverage_ok_early = (
    primary_pass["return_code"] == 0
    and len(primary_task_reports_early) == expected_task_count
    and set(primary_task_counter_early) == expected_tasks
    and all(count == 1 for count in primary_task_counter_early.values())
)
expected_basekey_count = sum(len(basekeys) for basekeys in expected_by_task.values())
primary_present_ratio = len(primary_quality["present_basekeys"]) / max(expected_basekey_count, 1)
post_primary_reasons = []
if primary_pass["return_code"] != 0:
    post_primary_reasons.append("PRIMARY_PROCESS_RETURN_CODE_NONZERO")
if not primary_worker_seed_ok_early:
    post_primary_reasons.append("PRIMARY_WORKER_SEED_AUDIT_FAILED")
if not primary_task_coverage_ok_early:
    post_primary_reasons.append("PRIMARY_CHALLENGE_SCOPE_COVERAGE_AUDIT_FAILED")
if primary_quality.get("scanner_ok") is not True:
    post_primary_reasons.append("PRIMARY_QUALITY_SCANNER_FAILED")
if primary_quality["corrupt_files"]:
    post_primary_reasons.append("PRIMARY_CORRUPT_OUTPUT_FILES")
if primary_present_ratio < MIN_PRIMARY_PRESENT_RATIO:
    post_primary_reasons.append("PRIMARY_PRESENT_OUTPUT_RATIO_BELOW_94_PERCENT")

# Freeze every primary byte before the independent deep pass starts.
default_output_dir.rename(primary_output_dir)
default_output_dir.mkdir(parents=True, exist_ok=True)

# Persist and byte-verify the expensive primary pass before any deep work. The
# uncompressed tar keeps already-compressed BZ2 files cheap to archive and makes
# a later recovery notebook possible without repeating roughly five GPU hours.
primary_checkpoint = create_verified_primary_checkpoint(
    primary_output_dir,
    run_plan["challenge_path"],
    actual_hashes,
)
if primary_checkpoint.get("ok") is not True:
    post_primary_reasons.append("PRIMARY_CHECKPOINT_VERIFICATION_FAILED")
print(
    "PROGRAM077_PRIMARY_CHECKPOINT_GATE="
    + ("OK" if primary_checkpoint.get("ok") is True else "FAILED")
)
print("PROGRAM077_PRIMARY_CHECKPOINT_REPORT=" + json.dumps(primary_checkpoint, sort_keys=True))

deep_status = "SKIPPED_BY_PRIMARY_ONLY_POLICY"
deep_pass = None
deep_deadline = None
remaining_before_deep = run_plan["global_end_time"] - time.time()
if post_primary_reasons:
    post_primary_early_gate = "BLOCK"
    deep_status = "SKIPPED_POST_PRIMARY_EARLY_BLOCK"
else:
    post_primary_early_gate = "GO"
    post_primary_reasons = [
        "PRIMARY_PROCESS_OK",
        "PRIMARY_WORKER_SEED_AUDIT_OK",
        "PRIMARY_CHALLENGE_SCOPE_COVERAGE_OK",
        "PRIMARY_OUTPUT_FILES_PARSE_OK",
        "PRIMARY_PRESENT_OUTPUT_RATIO_AT_LEAST_94_PERCENT",
        "NORMAL_NONDETERMINISTIC_CUDA_BACKEND_ACTIVE",
        "PRIMARY_ONLY_POLICY_ACTIVE",
    ]

post_primary_early_report = {
    "gate": post_primary_early_gate,
    "reasons": post_primary_reasons,
    "primary_return_code": primary_pass["return_code"],
    "primary_worker_seed_ok": primary_worker_seed_ok_early,
    "primary_task_coverage_ok": primary_task_coverage_ok_early,
    "primary_present_basekey_count": len(primary_quality["present_basekeys"]),
    "expected_basekey_count": expected_basekey_count,
    "primary_present_ratio": round(primary_present_ratio, 6),
    "primary_corrupt_file_count": len(primary_quality["corrupt_files"]),
    "primary_quality_scanner_ok": bool(primary_quality.get("scanner_ok")),
    "primary_quality_scanner_python": primary_quality.get("scanner_python"),
    "deep_task_count": 0,
    "remaining_before_deep_seconds": round(remaining_before_deep, 3),
}
print(f"PROGRAM077_POST_PRIMARY_EARLY_GATE={post_primary_early_gate}")
print("PROGRAM077_POST_PRIMARY_EARLY_REPORT=" + json.dumps(post_primary_early_report, sort_keys=True))

if deep_output_dir.exists():
    shutil.rmtree(deep_output_dir)
if default_output_dir.exists():
    default_output_dir.rename(deep_output_dir)
else:
    deep_output_dir.mkdir(parents=True, exist_ok=True)
default_output_dir.mkdir(parents=True, exist_ok=True)

deep_quality = snapshot_output_quality_isolated(
    deep_output_dir, expected_by_task, "deep", run_env
)
primary_missing_set = set(primary_quality["missing_basekeys"])
primary_starved_set = set(primary_quality["starved_basekeys"])
deep_present_set = set(deep_quality["present_basekeys"])
recovered_basekeys = sorted(primary_missing_set & deep_present_set)
deep_eligible_present_basekeys = sorted(primary_starved_set & deep_present_set)
final_missing_basekeys = sorted(primary_missing_set - deep_present_set)
ignored_deep_basekeys = sorted(deep_present_set - primary_starved_set)
starter_log_text = starter_log_path.read_text(encoding="utf-8", errors="replace")
starter_log_sha256 = hashlib.sha256(starter_log_text.encode("utf-8")).hexdigest()
worker_seed_reports = parse_json_markers(starter_log_text, "PROGRAM077_WORKER_SEED_READY")
task_seed_reports = parse_json_markers(starter_log_text, "PROGRAM077_TASK_SEED_READY")
seed_audit_errors = []
for report in worker_seed_reports + task_seed_reports:
    label = report.get("run_label")
    expected_seed = 42 if label == "primary" else None
    if expected_seed is None or report.get("base_seed") != expected_seed:
        seed_audit_errors.append({"reason": "base_seed", "report": report})
    if report.get("python_hash_seed") != "0":
        seed_audit_errors.append({"reason": "python_hash_seed", "report": report})
    if report.get("seed_policy") != "rng_only_no_backend_change":
        seed_audit_errors.append({"reason": "seed_policy", "report": report})
    if report.get("deterministic_algorithms_enabled") is not False:
        seed_audit_errors.append({"reason": "backend_policy", "report": report})
if seed_audit_errors:
    raise RuntimeError("PROGRAM077_SEED_AUDIT_FAILED=" + json.dumps(seed_audit_errors, sort_keys=True))
hash_probes = {report.get("hash_probe") for report in worker_seed_reports + task_seed_reports}
if len(hash_probes) > 1:
    raise RuntimeError(f"PROGRAM077_HASH_PROBE_MISMATCH={sorted(hash_probes)}")

primary_worker_reports = [r for r in worker_seed_reports if r.get("run_label") == "primary"]
deep_worker_reports = [r for r in worker_seed_reports if r.get("run_label") == "deep_retry"]
primary_task_reports = [r for r in task_seed_reports if r.get("run_label") == "primary"]
deep_task_reports = [r for r in task_seed_reports if r.get("run_label") == "deep_retry"]
primary_task_counter = Counter(r.get("task_id") for r in primary_task_reports)
deep_task_counter = Counter(r.get("task_id") for r in deep_task_reports)
primary_worker_seed_ok = sorted(r.get("rank") for r in primary_worker_reports) == [0, 1, 2, 3]
deep_worker_seed_ok = (
    deep_pass is None
    or sorted(r.get("rank") for r in deep_worker_reports) == [0, 1, 2, 3]
)
primary_task_coverage_ok = (
    primary_pass["return_code"] == 0
    and len(primary_task_reports) == expected_task_count
    and set(primary_task_counter) == expected_tasks
    and all(count == 1 for count in primary_task_counter.values())
)
deep_task_scope_ok = (
    set(deep_task_counter).issubset(set(deep_tasks))
    and all(count == 1 for count in deep_task_counter.values())
)

training_pattern = re.compile(
    r"^\[Rank (?P<rank>\d+)\] training stats for puzzle (?P<task>[0-9a-f]+): "
    r"TrainOutput\(global_step=(?P<global_step>\d+), training_loss=(?P<training_loss>[-+0-9.eE]+)",
    flags=re.MULTILINE,
)
training_records = [
    {
        "rank": int(match.group("rank")),
        "task_id": match.group("task"),
        "global_step": int(match.group("global_step")),
        "training_loss": float(match.group("training_loss")),
    }
    for match in training_pattern.finditer(starter_log_text)
]

forecast_seconds = forecast_runtime_seconds(len(deep_tasks))
recovery_report = {
    "program_no": PROGRAM_NO,
    "policy": "normal_cuda_program024_primary_only",
    "primary_status": primary_status,
    "deep_status": deep_status,
    "early_format_reload_selftest": early_format_selftest,
    "candidate_pickle_fixture": candidate_pickle_fixture,
    "post_primary_early_gate": post_primary_early_report,
    "primary_checkpoint": primary_checkpoint,
    "deep_skipped_by_early_gate": True,
    "minimum_primary_present_ratio": MIN_PRIMARY_PRESENT_RATIO,
    "max_deep_tasks": MAX_DEEP_TASKS,
    "deep_wall_limit_seconds": DEEP_WALL_LIMIT_SECONDS,
    "primary_output_dir": str(primary_output_dir),
    "deep_output_dir": str(deep_output_dir),
    "primary": primary_quality,
    "deep": deep_quality,
    "deep_tasks": deep_tasks,
    "deep_task_count": len(deep_tasks),
    "deep_deadline": deep_deadline,
    "recovered_basekeys": recovered_basekeys,
    "recovered_basekey_count": len(recovered_basekeys),
    "deep_eligible_present_basekeys": deep_eligible_present_basekeys,
    "deep_eligible_present_basekey_count": len(deep_eligible_present_basekeys),
    "final_missing_basekeys": final_missing_basekeys,
    "final_missing_basekey_count": len(final_missing_basekeys),
    "ignored_deep_basekeys": ignored_deep_basekeys,
    "ignored_deep_basekey_count": len(ignored_deep_basekeys),
    "primary_worker_seed_ok": primary_worker_seed_ok,
    "deep_worker_seed_ok": deep_worker_seed_ok,
    "primary_task_coverage_ok": primary_task_coverage_ok,
    "deep_task_scope_ok": deep_task_scope_ok,
    "primary_pass": primary_pass,
    "deep_pass": deep_pass,
    "runtime_forecast": {
        "hardware": "NVIDIA_L4_X4",
        "primary_reference_seconds": PRIMARY_RUNTIME_REFERENCE_SECONDS,
        "primary_checkpoint_reference_seconds": PRIMARY_CHECKPOINT_REFERENCE_SECONDS,
        "deep_task_reference_seconds": DEEP_TASK_REFERENCE_SECONDS,
        "deep_process_overhead_seconds": DEEP_PROCESS_OVERHEAD_SECONDS,
        "deep_task_count": len(deep_tasks),
        "forecast_total_seconds": forecast_seconds,
        "forecast_total_hours": round(forecast_seconds / 3600, 3),
        "typical_range_text": "5h10m-9h15m_primary_plus_15m_finalization",
        "hard_notebook_limit_seconds": 9 * 3600 + 30 * 60,
    },
}
recovery_report_path = Path("/kaggle/working/program077_recovery_report.json")
recovery_report_path.write_text(
    json.dumps(recovery_report, sort_keys=True, separators=(",", ":")),
    encoding="utf-8",
)

diagnostics = {
    "program_no": PROGRAM_NO,
    "seed_contract": run_plan["seed_contract"],
    "deep_base_seed": DEEP_BASE_SEED,
    "early_format_reload_selftest": early_format_selftest,
    "candidate_pickle_fixture": candidate_pickle_fixture,
    "post_primary_early_gate": post_primary_early_report,
    "primary_checkpoint": primary_checkpoint,
    "starter_log": str(starter_log_path),
    "starter_log_sha256": starter_log_sha256,
    "worker_seed_reports": worker_seed_reports,
    "task_seed_reports": task_seed_reports,
    "training_records": training_records,
    "primary_task_counter": dict(sorted(primary_task_counter.items())),
    "deep_task_counter": dict(sorted(deep_task_counter.items())),
    "primary_task_coverage_ok": primary_task_coverage_ok,
    "deep_task_scope_ok": deep_task_scope_ok,
    "recovery_report": str(recovery_report_path),
}
diagnostics_path = Path("/kaggle/working/program077_run_diagnostics.json")
diagnostics_path.write_text(
    json.dumps(diagnostics, sort_keys=True, separators=(",", ":")),
    encoding="utf-8",
)
diagnostics_sha256 = hashlib.sha256(diagnostics_path.read_bytes()).hexdigest()
print("PROGRAM077_PRIMARY_ONLY_PLAN=" + json.dumps({
    "primary_missing_basekeys": primary_quality["missing_basekeys"],
    "primary_starved_basekeys": primary_quality["starved_basekeys"],
    "deep_tasks": deep_tasks,
    "deep_status": deep_status,
    "post_primary_early_gate": post_primary_early_gate,
    "forecast_total_seconds": forecast_seconds,
}, sort_keys=True))
print("PROGRAM077_PRIMARY_ONLY_RESULT=" + json.dumps({
    "primary_missing_count": len(primary_quality["missing_basekeys"]),
    "primary_starved_count": len(primary_quality["starved_basekeys"]),
    "recovered_count": len(recovered_basekeys),
    "deep_eligible_present_count": len(deep_eligible_present_basekeys),
    "final_missing_count": len(final_missing_basekeys),
    "ignored_deep_existing_basekeys": len(ignored_deep_basekeys),
}, sort_keys=True))
print("PROGRAM077_SEED_AUDIT=" + json.dumps({
    "hash_probe": next(iter(hash_probes)) if hash_probes else None,
    "primary_worker_seed_count": len(primary_worker_reports),
    "deep_worker_seed_count": len(deep_worker_reports),
    "primary_task_seed_count": len(primary_task_reports),
    "deep_task_seed_count": len(deep_task_reports),
    "primary_worker_seed_ok": primary_worker_seed_ok,
    "deep_worker_seed_ok": deep_worker_seed_ok,
    "primary_task_coverage_ok": primary_task_coverage_ok,
    "deep_task_scope_ok": deep_task_scope_ok,
}, sort_keys=True))
print(f"PROGRAM077_RUN_DIAGNOSTICS_SHA256={diagnostics_sha256}")
print("PROGRAM077_SINGLE_PIPE_LOGGING_OK=" + json.dumps({
    "starter_log": str(starter_log_path),
    "starter_log_sha256": starter_log_sha256,
    "primary_return_code": primary_pass["return_code"],
    "deep_return_code": deep_pass["return_code"] if deep_pass else None,
}, sort_keys=True))


In [ ]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path

run_plan = json.loads(Path("program077_run_plan.json").read_text(encoding="utf-8"))
legacy_root = Path(run_plan["legacy_root"]).resolve()
overlay_root = Path(run_plan["overlay_root"]).resolve()
python_executable = Path(run_plan["python_executable"]).resolve()

finalize_path = Path("/kaggle/working/program077_finalize.py")
bootstrap_source = """import os
import sys
from pathlib import Path
legacy_root = Path(os.environ["PROGRAM077_LEGACY_ROOT"]).resolve()
overlay_root = Path(os.environ["PROGRAM077_OVERLAY_ROOT"]).resolve()
clean = []
for item in sys.path:
    if not item:
        clean.append(item)
        continue
    try:
        resolved = Path(item).resolve()
    except OSError:
        continue
    if resolved == legacy_root or legacy_root in resolved.parents or resolved == overlay_root:
        continue
    clean.append(item)
sys.path[:] = [str(overlay_root)] + clean
"""

finalize_body = r'''
import bz2
import hashlib
import json
import math
import os
import pickle
import re
import time
from collections import Counter, defaultdict
from fractions import Fraction
from pathlib import Path

import numpy as np
from arc_loader import ArcDataset
from arc_decoder import hashable, score_kgmon, score_full_probmul_3


PROGRAM_NO = 77
POLICY_NAME = "normal_cuda_program024_primary_only_v2"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def grid_key(grid):
    array = np.asarray(grid)
    if array.ndim != 2 or not 1 <= array.shape[0] <= 30 or not 1 <= array.shape[1] <= 30:
        raise ValueError(f"invalid grid shape {array.shape}")
    if not np.issubdtype(array.dtype, np.integer):
        if not np.all(np.equal(array, np.floor(array))):
            raise ValueError("grid contains non-integer values")
    array = array.astype(int)
    if np.any(array < 0) or np.any(array > 9):
        raise ValueError("grid values must be in 0..9")
    return tuple(tuple(int(value) for value in row) for row in array)


def _validate_grid(grid, location: str, errors: list[str]) -> None:
    try:
        grid_key(grid)
    except Exception as exc:
        errors.append(f"{location}: {type(exc).__name__}: {exc}")


def validate_submission_against_challenge(challenge: dict, submission: dict, label: str) -> dict:
    errors = []
    if not isinstance(challenge, dict) or not challenge:
        errors.append("challenge must be a non-empty object")
    if not isinstance(submission, dict) or not submission:
        errors.append("submission must be a non-empty object")
    if errors:
        return {"label": label, "ok": False, "errors": errors}
    if set(challenge) != set(submission):
        errors.append("submission task ids do not exactly match challenge")
    checked_tests = 0
    checked_attempts = 0
    for task_id in sorted(set(challenge) & set(submission)):
        expected_tests = challenge[task_id].get("test", [])
        items = submission[task_id]
        if not isinstance(items, list) or len(items) != len(expected_tests):
            errors.append(f"{task_id}: incorrect test-output count")
            continue
        for test_index, item in enumerate(items):
            checked_tests += 1
            location = f"{task_id}[{test_index}]"
            if not isinstance(item, dict) or set(item) != {"attempt_1", "attempt_2"}:
                errors.append(f"{location}: attempts must be exactly attempt_1 and attempt_2")
                continue
            for attempt in ("attempt_1", "attempt_2"):
                checked_attempts += 1
                _validate_grid(item[attempt], f"{location}.{attempt}", errors)
    return {
        "label": label,
        "ok": not errors,
        "challenge_task_count": len(challenge),
        "submission_task_count": len(submission),
        "checked_test_count": checked_tests,
        "checked_attempt_count": checked_attempts,
        "error_count": len(errors),
        "errors": errors[:50],
    }


def safe_load_decoded(root: Path, source: str):
    decoded = defaultdict(dict)
    corrupt_files = []
    file_basekeys = set()
    if not root.is_dir():
        return {}, corrupt_files, file_basekeys
    for path in sorted(item for item in root.iterdir() if item.is_file()):
        basekey = path.name.split(".", 1)[0]
        file_basekeys.add(basekey)
        try:
            with bz2.BZ2File(path) as handle:
                outputs = pickle.load(handle)
            if not isinstance(outputs, list):
                raise TypeError(f"payload must be list, got {type(outputs).__name__}")
            staged = {}
            for index, sample in enumerate(outputs):
                if not isinstance(sample, dict) or not {"solution", "beam_score", "score_aug"}.issubset(sample):
                    continue
                grid_key(sample["solution"])
                staged[f"{source}::{path.name}.out{index}"] = sample
            # A file is atomic: no candidate from it is admitted if any later
            # candidate makes the payload invalid.
            decoded[basekey].update(staged)
        except Exception as exc:
            corrupt_files.append({
                "source": source,
                "name": path.name,
                "error": f"{type(exc).__name__}: {exc}",
            })
    return {key: dict(values) for key, values in decoded.items()}, corrupt_files, file_basekeys


def unique_candidate_count(records):
    fingerprints = set()
    for sample in records.values():
        try:
            fingerprints.add(grid_key(sample["solution"]))
        except Exception:
            continue
    return len(fingerprints)


def task_and_test_index(basekey):
    task_id, test_index = basekey.rsplit("_", 1)
    return task_id, int(test_index)


def exact_shape_predictions(task, test_index):
    train = task.get("train", [])
    if not train:
        return None, []
    pairs = [
        (tuple(np.asarray(pair["input"]).shape), tuple(np.asarray(pair["output"]).shape))
        for pair in train
    ]
    test_shape = tuple(np.asarray(task["test"][test_index]["input"]).shape)
    predictions = set()
    rules = []
    if all(output_shape == input_shape for input_shape, output_shape in pairs):
        predictions.add(test_shape)
        rules.append("same_as_input")
    if all(output_shape == input_shape[::-1] for input_shape, output_shape in pairs):
        predictions.add(test_shape[::-1])
        rules.append("transpose_shape")
    output_shapes = {output_shape for _, output_shape in pairs}
    if len(output_shapes) == 1:
        predictions.update(output_shapes)
        rules.append("constant_output_shape")
    row_ratios = {Fraction(output_shape[0], input_shape[0]) for input_shape, output_shape in pairs}
    col_ratios = {Fraction(output_shape[1], input_shape[1]) for input_shape, output_shape in pairs}
    if len(row_ratios) == 1 and len(col_ratios) == 1:
        row_ratio = next(iter(row_ratios))
        col_ratio = next(iter(col_ratios))
        predicted_rows = Fraction(test_shape[0]) * row_ratio
        predicted_cols = Fraction(test_shape[1]) * col_ratio
        if predicted_rows.denominator == 1 and predicted_cols.denominator == 1:
            predicted = (int(predicted_rows), int(predicted_cols))
            if 1 <= predicted[0] <= 30 and 1 <= predicted[1] <= 30:
                predictions.add(predicted)
                rules.append("exact_rational_scale")
    return (predictions if predictions else None), rules


def allowed_palette(task, test_index):
    colors = set()
    for pair in task.get("train", []):
        colors.update(int(value) for value in np.asarray(pair["input"]).ravel())
        colors.update(int(value) for value in np.asarray(pair["output"]).ravel())
    colors.update(int(value) for value in np.asarray(task["test"][test_index]["input"]).ravel())
    return colors


def structural_check(solution, task, test_index):
    try:
        fingerprint = grid_key(solution)
    except Exception as exc:
        return False, [f"grid:{type(exc).__name__}"]
    array = np.asarray(fingerprint, dtype=int)
    reasons = []
    palette = allowed_palette(task, test_index)
    extra_colors = sorted(set(int(value) for value in array.ravel()) - palette)
    if extra_colors:
        reasons.append("unseen_colors:" + "-".join(map(str, extra_colors)))
    predicted_shapes, shape_rules = exact_shape_predictions(task, test_index)
    if predicted_shapes is not None and tuple(array.shape) not in predicted_shapes:
        reasons.append("shape_outside_exact_train_schema")
    return not reasons, reasons


def geometry_family(sample_key):
    raw_key = sample_key.split("::", 1)[-1]
    operations = raw_key.split(".")[1:]
    transpose_parity = sum(operation == "transpose" for operation in operations) % 2
    rotation = sum(operation == "rot90" for operation in operations) % 4
    return f"t{transpose_parity}_r{rotation}"


def finite_mean(values, default=1e12):
    finite = []
    for value in values:
        try:
            number = float(value)
        except Exception:
            continue
        if math.isfinite(number):
            finite.append(number)
    return float(np.mean(finite)) if finite else float(default)


def rank_structural_evidence(records, task, test_index):
    groups = {}
    for sample_key, sample in records.items():
        fingerprint = grid_key(sample["solution"])
        group = groups.setdefault(fingerprint, {
            "solution": np.asarray(fingerprint, dtype=int),
            "records": [],
            "sources": set(),
            "families": set(),
            "origins": set(),
            "source_counts": Counter(),
        })
        source = sample_key.split("::", 1)[0] if "::" in sample_key else "unknown"
        origin = re.sub(r"\.out\d+$", "", sample_key)
        group["records"].append(sample)
        group["sources"].add(source)
        group["families"].add(geometry_family(sample_key))
        group["origins"].add(origin)
        group["source_counts"][source] += 1

    ranked = []
    for fingerprint, group in groups.items():
        valid, rejection_reasons = structural_check(group["solution"], task, test_index)
        augmented = [
            value
            for sample in group["records"]
            for value in np.asarray(sample.get("score_aug", [])).ravel().tolist()
        ]
        beams = [sample.get("beam_score") for sample in group["records"]]
        group.update({
            "fingerprint": fingerprint,
            "structural_valid": valid,
            "rejection_reasons": rejection_reasons,
            "source_count": len(group["sources"]),
            "family_count": len(group["families"]),
            "origin_count": len(group["origins"]),
            "balanced_support": sum(min(count, 8) / 8 for count in group["source_counts"].values()),
            "mean_aug_score": finite_mean(augmented),
            "mean_beam_score": finite_mean(beams),
        })
        ranked.append(group)

    structurally_valid = [group for group in ranked if group["structural_valid"]]
    fallback_used = not structurally_valid and bool(ranked)
    pool = structurally_valid if structurally_valid else ranked
    pool.sort(key=lambda group: (
        -group["source_count"],
        -group["family_count"],
        -min(group["origin_count"], 16),
        -group["balanced_support"],
        group["mean_aug_score"],
        group["mean_beam_score"],
        repr(group["fingerprint"]),
    ))
    audit = {
        "unique_candidate_count": len(ranked),
        "structurally_valid_count": len(structurally_valid),
        "structurally_rejected_count": len(ranked) - len(structurally_valid),
        "structural_fallback_used": fallback_used,
        "shape_rules": exact_shape_predictions(task, test_index)[1],
        "ranked": [
            {
                "grid_sha256": hashlib.sha256(repr(group["fingerprint"]).encode("utf-8")).hexdigest(),
                "shape": list(group["solution"].shape),
                "structural_valid": group["structural_valid"],
                "rejection_reasons": group["rejection_reasons"],
                "source_count": group["source_count"],
                "sources": sorted(group["sources"]),
                "family_count": group["family_count"],
                "origin_count": group["origin_count"],
                "balanced_support": group["balanced_support"],
                "mean_aug_score": group["mean_aug_score"],
                "mean_beam_score": group["mean_beam_score"],
            }
            for group in pool[:10]
        ],
    }
    return pool, audit


legacy_root = Path(os.environ["PROGRAM077_LEGACY_ROOT"]).resolve()
overlay_root = Path(os.environ["PROGRAM077_OVERLAY_ROOT"]).resolve()
numpy_path = Path(np.__file__).resolve()
if numpy_path == legacy_root or legacy_root in numpy_path.parents:
    raise RuntimeError(f"Finalize NumPy leaked from legacy root: {numpy_path}")
if numpy_path == overlay_root or overlay_root in numpy_path.parents:
    raise RuntimeError(f"Finalize NumPy leaked from overlay: {numpy_path}")

run_plan = json.loads(Path("program077_run_plan.json").read_text(encoding="utf-8"))
recovery_report_path = Path("program077_recovery_report.json")
diagnostics_path = Path("program077_run_diagnostics.json")
if not recovery_report_path.is_file() or not diagnostics_path.is_file():
    raise RuntimeError("PROGRAM077_RECOVERY_OR_DIAGNOSTICS_MISSING")
recovery_report = json.loads(recovery_report_path.read_text(encoding="utf-8"))
diagnostics = json.loads(diagnostics_path.read_text(encoding="utf-8"))
challenge_path = Path(run_plan["challenge_path"])
challenge_payload = json.loads(challenge_path.read_text(encoding="utf-8"))
rerun_mode = bool(run_plan["rerun_mode"])
submission_path = Path("/kaggle/working/submission.json")
submission_path.unlink(missing_ok=True)

data = ArcDataset.from_file(str(challenge_path))
if not rerun_mode:
    data = data.load_replies(str(Path(run_plan["solution_path"])))

primary_output_dir = Path(recovery_report["primary_output_dir"])
deep_output_dir = Path(recovery_report["deep_output_dir"])
primary_decoded, primary_corrupt_files, primary_file_basekeys = safe_load_decoded(primary_output_dir, "primary")
deep_decoded, deep_corrupt_files, deep_file_basekeys = safe_load_decoded(deep_output_dir, "deep")
corrupt_files = primary_corrupt_files + deep_corrupt_files

expected_basekeys = [
    f"{task_id}_{test_index}"
    for task_id, task in sorted(challenge_payload.items())
    for test_index in range(len(task.get("test", [])))
]
primary_unique_counts = {
    basekey: unique_candidate_count(primary_decoded.get(basekey, {}))
    for basekey in expected_basekeys
}
primary_missing_basekeys = {
    basekey for basekey, count in primary_unique_counts.items() if count == 0
}
eligible_basekeys = {
    basekey for basekey, count in primary_unique_counts.items() if count < 2
}
orchestrator_missing_basekeys = set(recovery_report["primary"].get("missing_basekeys", []))
orchestrator_starved_basekeys = set(recovery_report["primary"].get("starved_basekeys", []))
quality_scanner_consistent = (
    orchestrator_missing_basekeys == primary_missing_basekeys
    and orchestrator_starved_basekeys == eligible_basekeys
)
quality_scanner_mismatch = {
    "consistent": quality_scanner_consistent,
    "orchestrator_missing_count": len(orchestrator_missing_basekeys),
    "finalizer_missing_count": len(primary_missing_basekeys),
    "orchestrator_starved_count": len(orchestrator_starved_basekeys),
    "finalizer_starved_count": len(eligible_basekeys),
}

primary_results = {
    basekey: score_kgmon(dict(values))
    for basekey, values in primary_decoded.items()
    if values
}
program077_results = {
    basekey: list(guesses[:2])
    for basekey, guesses in primary_results.items()
}
evidence_audit = {}
used_deep_basekeys = set()

# Invariants: all covered primary outputs remain exact KGMon, and every existing
# primary top-1 remains byte-for-byte unchanged.
policy_violations = []
for basekey in expected_basekeys:
    baseline = primary_results.get(basekey, [])
    final = program077_results.get(basekey, [])
    if baseline and (not final or grid_key(baseline[0]) != grid_key(final[0])):
        policy_violations.append({"basekey": basekey, "reason": "primary_top1_changed"})
    if basekey not in eligible_basekeys:
        if [grid_key(value) for value in baseline[:2]] != [grid_key(value) for value in final[:2]]:
            policy_violations.append({"basekey": basekey, "reason": "well_covered_primary_changed"})
if policy_violations:
    raise RuntimeError("PROGRAM077_PRIMARY_POLICY_LOCK_FAILED=" + json.dumps(policy_violations, sort_keys=True))

baseline_submission = data.get_submission(primary_results)
experimental_submission = data.get_submission(program077_results)
if experimental_submission != baseline_submission:
    raise RuntimeError("PROGRAM077_PRIMARY_ONLY_SUBMISSION_DIVERGED_FROM_KGMON")
top1_changed_keys = []
second_slot_changed_keys = []
change_details = {}
for basekey in expected_basekeys:
    baseline = primary_results.get(basekey, [])
    final = program077_results.get(basekey, [])
    baseline_keys = [grid_key(value) for value in baseline[:2]]
    final_keys = [grid_key(value) for value in final[:2]]
    top1_changed = (baseline_keys[:1] != final_keys[:1])
    second_changed = (baseline_keys[1:2] != final_keys[1:2])
    if top1_changed:
        top1_changed_keys.append(basekey)
    if second_changed:
        second_slot_changed_keys.append(basekey)
    if top1_changed or second_changed:
        change_details[basekey] = {
            "primary_missing": basekey in primary_missing_basekeys,
            "primary_unique_candidate_count": primary_unique_counts[basekey],
            "top1_changed": top1_changed,
            "second_slot_changed": second_changed,
            "deep_candidate_used": basekey in used_deep_basekeys,
            "evidence": evidence_audit.get(basekey, {}),
        }
if not set(top1_changed_keys).issubset(primary_missing_basekeys):
    raise RuntimeError("PROGRAM077_EXISTING_PRIMARY_TOP1_CHANGED=" + json.dumps(top1_changed_keys))
if not set(top1_changed_keys + second_slot_changed_keys).issubset(eligible_basekeys):
    raise RuntimeError("PROGRAM077_CHANGE_OUTSIDE_STARVATION_SCOPE")

local_evaluation_max_points = "NA_COMPETITION_RERUN"
baseline_local_evaluation_raw_points = "NA_COMPETITION_RERUN"
program077_local_evaluation_raw_points = "NA_COMPETITION_RERUN"
baseline_local_evaluation_percent = "NA_COMPETITION_RERUN"
program077_local_evaluation_percent = "NA_COMPETITION_RERUN"
if not rerun_mode:
    local_evaluation_max_points = len(data.replies)
    if local_evaluation_max_points <= 0:
        raise RuntimeError("PROGRAM077_LOCAL_EVALUATION_DENOMINATOR_INVALID")
    baseline_local_evaluation_raw_points = data.validate_submission(baseline_submission)
    program077_local_evaluation_raw_points = data.validate_submission(experimental_submission)
    baseline_local_evaluation_percent = 100.0 * baseline_local_evaluation_raw_points / local_evaluation_max_points
    program077_local_evaluation_percent = 100.0 * program077_local_evaluation_raw_points / local_evaluation_max_points

primary_status_for_selection = recovery_report.get("primary_status", "MISSING")
deep_status_for_selection = recovery_report.get("deep_status", "MISSING")
post_primary_gate_for_selection = recovery_report.get("post_primary_early_gate", {}).get("gate", "MISSING")
checkpoint_ok_for_selection = recovery_report.get("primary_checkpoint", {}).get("ok") is True
selection_safety_fallback_reasons = []
if recovery_report.get("candidate_pickle_fixture", {}).get("ok") is not True:
    selection_safety_fallback_reasons.append("CANDIDATE_PICKLE_FIXTURE_NOT_VERIFIED")
if recovery_report.get("primary", {}).get("scanner_ok") is not True:
    selection_safety_fallback_reasons.append("PRIMARY_QUALITY_SCANNER_FAILED")
if recovery_report.get("deep", {}).get("scanner_ok") is not True:
    selection_safety_fallback_reasons.append("DEEP_QUALITY_SCANNER_FAILED")
if not quality_scanner_consistent:
    selection_safety_fallback_reasons.append("QUALITY_SCANNER_MISMATCH")
if post_primary_gate_for_selection != "GO":
    selection_safety_fallback_reasons.append("POST_PRIMARY_GATE_NOT_GO")
if primary_status_for_selection != "COMPLETED":
    selection_safety_fallback_reasons.append("PRIMARY_NOT_COMPLETED")
if deep_status_for_selection != "SKIPPED_BY_PRIMARY_ONLY_POLICY":
    selection_safety_fallback_reasons.append("PRIMARY_ONLY_DEEP_STATUS_INVALID")
if recovery_report.get("primary_task_coverage_ok") is not True:
    selection_safety_fallback_reasons.append("PRIMARY_TASK_COVERAGE_FAILED")
if recovery_report.get("primary_worker_seed_ok") is not True:
    selection_safety_fallback_reasons.append("PRIMARY_WORKER_SEED_FAILED")
if recovery_report.get("deep_worker_seed_ok") is not True:
    selection_safety_fallback_reasons.append("DEEP_WORKER_SEED_FAILED")
if recovery_report.get("deep_task_scope_ok") is not True:
    selection_safety_fallback_reasons.append("DEEP_TASK_SCOPE_FAILED")
if primary_corrupt_files:
    selection_safety_fallback_reasons.append("PRIMARY_CORRUPT_FILES")
if deep_corrupt_files:
    selection_safety_fallback_reasons.append("DEEP_CORRUPT_FILES")
if not checkpoint_ok_for_selection:
    selection_safety_fallback_reasons.append("PRIMARY_CHECKPOINT_NOT_VERIFIED")

def choose_program077_submission(rerun_mode, reasons, baseline_submission):
    reasons = list(reasons)
    competition_rerun_recovery_mode = bool(rerun_mode and reasons)
    if competition_rerun_recovery_mode:
        selected_policy = "PROGRAM077_HIDDEN_RECOVERED_PRIMARY"
    elif reasons:
        selected_policy = "PRIMARY_KGMON_SAFETY_FALLBACK"
    else:
        selected_policy = "PROGRAM077_NORMAL_CUDA_PRIMARY"
    return baseline_submission, selected_policy, competition_rerun_recovery_mode


selected_submission, selected_policy, competition_rerun_recovery_mode = choose_program077_submission(
    rerun_mode,
    selection_safety_fallback_reasons,
    baseline_submission,
)
if competition_rerun_recovery_mode:
    print(
        "PROGRAM077_COMPETITION_RERUN_RECOVERY="
        + json.dumps(selection_safety_fallback_reasons, sort_keys=True)
    )

submission_path = Path("/kaggle/working/submission.json")
baseline_path = Path("/kaggle/working/submission_program077_primary_kgmon.json")
experimental_path = Path("/kaggle/working/submission_program077_normal_cuda_primary.json")
format_report_path = Path("/kaggle/working/program077_submission_format_report.json")
for stale in (submission_path, baseline_path, experimental_path, format_report_path):
    stale.unlink(missing_ok=True)

format_reports = [
    validate_submission_against_challenge(challenge_payload, baseline_submission, "primary_kgmon"),
    validate_submission_against_challenge(challenge_payload, experimental_submission, "program077_normal_cuda_primary"),
    validate_submission_against_challenge(challenge_payload, selected_submission, "selected_submission"),
]
format_report = {
    "program_no": PROGRAM_NO,
    "challenge_path": str(challenge_path),
    "rerun_mode": rerun_mode,
    "selected_policy": selected_policy,
    "selection_safety_fallback_reasons": selection_safety_fallback_reasons,
    "reports": format_reports,
    "ok": all(report["ok"] for report in format_reports),
}
if not format_report["ok"]:
    format_report_path.write_text(json.dumps(format_report, sort_keys=True), encoding="utf-8")
    print("PROGRAM077_SUBMISSION_FORMAT_GATE=FAILED")
    print("PROGRAM077_SUBMISSION_FORMAT_REPORT=" + json.dumps(format_report, sort_keys=True))
    raise RuntimeError("PROGRAM077_SUBMISSION_FORMAT_GATE_FAILED")

def write_submission(path, payload):
    path.write_text(
        json.dumps(payload, sort_keys=True, separators=(",", ":"), allow_nan=False),
        encoding="utf-8",
    )


write_submission(baseline_path, baseline_submission)
write_submission(experimental_path, experimental_submission)
write_submission(submission_path, selected_submission)
reloaded_submission = json.loads(submission_path.read_text(encoding="utf-8"))
post_write_report = validate_submission_against_challenge(
    challenge_payload, reloaded_submission, "submission_json_reloaded"
)
if not post_write_report["ok"] or reloaded_submission != selected_submission:
    raise RuntimeError("PROGRAM077_POST_WRITE_FORMAT_GATE_FAILED=" + json.dumps(post_write_report, sort_keys=True))
format_report["post_write_report"] = post_write_report
format_report["ok"] = True
format_report_path.write_text(
    json.dumps(format_report, sort_keys=True, separators=(",", ":")),
    encoding="utf-8",
)

union_decoded = {}
for basekey in expected_basekeys:
    values = {}
    values.update(primary_decoded.get(basekey, {}))
    if basekey in eligible_basekeys:
        values.update(deep_decoded.get(basekey, {}))
    union_decoded[basekey] = values

candidate_starvation_count = 0
top1_ranker_disagreement_count = 0
quality_rows = []
for basekey in expected_basekeys:
    basevalues = union_decoded.get(basekey, {})
    unique_count = unique_candidate_count(basevalues)
    kgmon = score_kgmon(dict(basevalues)) if basevalues else []
    probmul = score_full_probmul_3(dict(basevalues)) if basevalues else []
    top1_agreement = bool(kgmon and probmul and hashable(kgmon[0]) == hashable(probmul[0]))
    candidate_starvation_count += int(unique_count < 2)
    top1_ranker_disagreement_count += int(not top1_agreement)
    quality_rows.append({
        "basekey": basekey,
        "primary_unique_candidate_count": primary_unique_counts[basekey],
        "combined_unique_candidate_count": unique_count,
        "top1_ranker_agreement": top1_agreement,
        "primary_missing": basekey in primary_missing_basekeys,
        "deep_eligible": basekey in eligible_basekeys,
        "deep_selected": basekey in used_deep_basekeys,
        "top1_changed": basekey in top1_changed_keys,
        "second_slot_changed": basekey in second_slot_changed_keys,
    })
final_populated_basekeys = {
    basekey for basekey, guesses in program077_results.items() if guesses
}
final_missing_basekeys = sorted(set(expected_basekeys) - final_populated_basekeys)
quality_summary = {
    "expected_test_output_count": len(expected_basekeys),
    "primary_missing_candidate_output_count": len(primary_missing_basekeys),
    "primary_starvation_count": len(eligible_basekeys),
    "deep_selected_basekey_count": len(used_deep_basekeys),
    "final_missing_candidate_output_count": len(final_missing_basekeys),
    "combined_candidate_starvation_count": candidate_starvation_count,
    "top1_ranker_disagreement_count": top1_ranker_disagreement_count,
    "top1_change_count": len(top1_changed_keys),
    "second_slot_change_count": len(second_slot_changed_keys),
}
quality_path = Path("program077_candidate_quality_atlas.json")
quality_path.write_text(
    json.dumps({"program_no": PROGRAM_NO, "summary": quality_summary, "keys": quality_rows}, sort_keys=True, separators=(",", ":")),
    encoding="utf-8",
)

output_entries = []
for source_label, root in (("primary", primary_output_dir), ("deep", deep_output_dir)):
    if not root.is_dir():
        continue
    for path in sorted(item for item in root.iterdir() if item.is_file()):
        basekey = path.name.split(".", 1)[0]
        output_entries.append({
            "source": source_label,
            "task_id": path.name.split("_", 1)[0],
            "basekey": basekey,
            "name": path.name,
            "size": path.stat().st_size,
            "sha256": sha256_file(path),
            "used_in_submission": source_label == "primary" or basekey in used_deep_basekeys,
        })
manifest = {
    "program_no": PROGRAM_NO,
    "variant": POLICY_NAME,
    "primary_file_count": sum(entry["source"] == "primary" for entry in output_entries),
    "deep_file_count": sum(entry["source"] == "deep" for entry in output_entries),
    "used_deep_file_count": sum(entry["source"] == "deep" and entry["used_in_submission"] for entry in output_entries),
    "files": output_entries,
}
manifest_path = Path("program077_output_manifest.json")
manifest_path.write_text(
    json.dumps(manifest, sort_keys=True, separators=(",", ":")),
    encoding="utf-8",
)
ranker_audit = {
    "program_no": PROGRAM_NO,
    "policy": POLICY_NAME,
    "quality_scanner_mismatch": quality_scanner_mismatch,
    "eligible_basekeys": sorted(eligible_basekeys),
    "used_deep_basekeys": sorted(used_deep_basekeys),
    "top1_changed_count": len(top1_changed_keys),
    "top1_changed_keys": top1_changed_keys,
    "second_slot_changed_count": len(second_slot_changed_keys),
    "second_slot_changed_keys": second_slot_changed_keys,
    "change_details": change_details,
}
ranker_audit_path = Path("program077_ranker_audit.json")
ranker_audit_path.write_text(
    json.dumps(ranker_audit, sort_keys=True, separators=(",", ":")),
    encoding="utf-8",
)

submission_sha256 = sha256_file(submission_path)
baseline_sha256 = sha256_file(baseline_path)
experimental_sha256 = sha256_file(experimental_path)
deep_solver_sha256 = "NOT_APPLICABLE_PRIMARY_ONLY"
reproducibility_payload = {
    "program_no": PROGRAM_NO,
    "variant": POLICY_NAME,
    "selected_policy": selected_policy,
    "selection_safety_fallback_reasons": selection_safety_fallback_reasons,
    "quality_scanner_mismatch": quality_scanner_mismatch,
    "seed_contract": run_plan["seed_contract"],
    "deep_seed_contract": None,
    "exact_program024_source_sha256": {
        "arc_loader.py": "d01cd56167e534ae156706ef62ab11662fc7b403964614a7a55131940bde970c",
        "arc_decoder.py": "965cfd910777d9bbec9681c6c15c5e2ea3924569734248046f5e58f16cb28222",
        "arc_solver.py": "f9011b4d4549fb688488d67ba7c2dc43184824436d34e1a4c2eae06f69daab73",
    },
    "deep_solver_sha256": deep_solver_sha256,
    "run_diagnostics_sha256": sha256_file(diagnostics_path),
    "recovery_report_sha256": sha256_file(recovery_report_path),
    "output_manifest_sha256": sha256_file(manifest_path),
    "ranker_audit_sha256": sha256_file(ranker_audit_path),
    "candidate_quality_atlas_sha256": sha256_file(quality_path),
    "submission_format_report_sha256": sha256_file(format_report_path),
    "baseline_submission_sha256": baseline_sha256,
    "experimental_submission_sha256": experimental_sha256,
    "submission_sha256": submission_sha256,
}
reproducibility_path = Path("program077_reproducibility_signature.json")
reproducibility_path.write_text(
    json.dumps(reproducibility_payload, sort_keys=True, separators=(",", ":")),
    encoding="utf-8",
)
reproducibility_sha256 = sha256_file(reproducibility_path)

primary_status = recovery_report["primary_status"]
deep_status = recovery_report["deep_status"]
primary_coverage_ok = bool(recovery_report.get("primary_task_coverage_ok"))
primary_checkpoint_report = recovery_report.get("primary_checkpoint", {})
candidate_pickle_fixture_report = recovery_report.get("candidate_pickle_fixture", {})
early_format_report = recovery_report.get("early_format_reload_selftest", {})
post_primary_early_report = recovery_report.get("post_primary_early_gate", {})
post_primary_early_gate = post_primary_early_report.get("gate", "MISSING")
post_primary_early_reasons = list(post_primary_early_report.get("reasons", []))
if competition_rerun_recovery_mode:
    daily_gate = "RECOVERED_FOR_HIDDEN_SCORING"
    daily_reasons = ["HIDDEN_RERUN_VALID_PRIMARY_FALLBACK_WRITTEN"] + selection_safety_fallback_reasons
    next_action = "HIDDEN_RERUN_SUBMISSION_WRITTEN_WITH_AUDITED_PARTIAL_PRIMARY"
elif post_primary_early_gate == "BLOCK":
    daily_gate = "BLOCK"
    daily_reasons = ["POST_PRIMARY_EARLY_GATE_BLOCK"] + post_primary_early_reasons
    next_action = "DO_NOT_SUBMIT_PASTE_GPT_FB_FOR_REVIEW"
elif not quality_scanner_consistent:
    daily_gate = "BLOCK"
    daily_reasons = ["ORCHESTRATOR_FINALIZER_QUALITY_SCAN_MISMATCH"]
    next_action = "DO_NOT_SUBMIT_PASTE_GPT_FB_FOR_REVIEW"
elif primary_status != "COMPLETED" or not primary_coverage_ok or primary_corrupt_files:
    daily_gate = "BLOCK"
    daily_reasons = ["PRIMARY_PASS_OR_COVERAGE_AUDIT_FAILED"]
    next_action = "DO_NOT_SUBMIT_PASTE_GPT_FB_FOR_REVIEW"
elif primary_checkpoint_report.get("ok") is not True:
    daily_gate = "BLOCK"
    daily_reasons = ["PRIMARY_CHECKPOINT_VERIFICATION_FAILED"]
    next_action = "DO_NOT_SUBMIT_PASTE_GPT_FB_FOR_REVIEW"
elif deep_status != "SKIPPED_BY_PRIMARY_ONLY_POLICY" or deep_corrupt_files:
    daily_gate = "BLOCK"
    daily_reasons = ["PRIMARY_ONLY_DEEP_STATE_INVALID"]
    next_action = "DO_NOT_SUBMIT_PASTE_GPT_FB_FOR_REVIEW"
elif selection_safety_fallback_reasons:
    daily_gate = "BLOCK"
    daily_reasons = ["FAIL_CLOSED_SELECTION_SAFETY_AUDIT"] + selection_safety_fallback_reasons
    next_action = "DO_NOT_SUBMIT_PASTE_GPT_FB_FOR_REVIEW"
elif baseline_submission != experimental_submission or selected_submission != baseline_submission:
    daily_gate = "BLOCK"
    daily_reasons = ["PRIMARY_ONLY_SUBMISSION_EQUALITY_FAILED"]
    next_action = "DO_NOT_SUBMIT_PASTE_GPT_FB_FOR_REVIEW"
else:
    daily_gate = "GO_FOR_USER_REVIEW"
    daily_reasons = [
        "FORMAT_AND_POST_WRITE_RELOAD_OK",
        "NORMAL_NONDETERMINISTIC_CUDA_BACKEND_ACTIVE",
        "PRIMARY_CHALLENGE_SCOPE_COVERAGE_OK",
        "PRIMARY_CHECKPOINT_VERIFIED",
        "EXACT_KGMON_TWO_GUESSES_LOCKED",
        "NO_DEEP_OR_POSTHOC_CHANGE",
        "LOCAL_EVALUATION_DIAGNOSTIC_ONLY_NOT_A_GATE",
    ]
    next_action = "REVIEW_GPT_FB_THEN_DECIDE_ONE_MANUAL_SUBMISSION"

task_counts = Counter(entry["task_id"] for entry in output_entries if entry["used_in_submission"])
notebook_runtime_seconds = round(time.time() - float(run_plan["notebook_start_time"]), 3)
if competition_rerun_recovery_mode:
    first_error = "RECOVERED_HIDDEN_RERUN_QUALITY_AUDIT"
elif post_primary_early_gate == "BLOCK":
    first_error = "POST_PRIMARY_EARLY_GATE_BLOCK"
elif not quality_scanner_consistent:
    first_error = "QUALITY_SCANNER_CONSISTENCY_GATE_FAILED"
elif selection_safety_fallback_reasons:
    first_error = "FAIL_CLOSED_SELECTION_SAFETY_AUDIT_FAILED"
elif primary_status != "COMPLETED" or deep_status == "FAILED_RECOVERABLE":
    first_error = f"PRIMARY_{primary_status}_DEEP_{deep_status}"
else:
    first_error = "NONE"
gpt_fb_lines = [
    "＝＝＝＝GPT FB START ＝＝＝＝",
    "GPT_FB_VERSION: 21",
    "PROGRAM_NO: 077",
    "BUILD_REVISION: 1_HIDDEN_RERUN_RECOVERY_PRIMARY_UNDER_10H",
    "PURPOSE: Preserve exact Program024 Normal-CUDA Primary while converting recoverable hidden quality-audit failures into a complete schema-valid submission under ten hours.",
    "EXECUTION_STATUS: " + ("COMPLETED_WITH_HIDDEN_RECOVERY" if competition_rerun_recovery_mode else "COMPLETED"),
    "FIRST_ERROR: " + first_error,
    "SUBMIT_POLICY: NEVER_AUTO_SUBMIT_USER_REVIEW_REQUIRED",
    f"RUNTIME_FAMILY: {run_plan.get('runtime_family')}",
    "DATA_SCOPE: " + ("COMPETITION_RERUN_HIDDEN_TEST" if rerun_mode else "VISIBLE_EVALUATION_DIAGNOSTIC"),
    "EXTERNAL_ASSETS: NONE_BEYOND_EXISTING_PROGRAM024_STACK",
    "SOURCE_HASH_CHECK: EXACT_PROGRAM024_PRIMARY_OK",
    "EARLY_GPU_GATE: L4_X4_OK",
    "BACKEND_POLICY: NORMAL_NONDETERMINISTIC_CUDA_BACKWARD",
    "DEEP_POLICY: DISABLED_PRIMARY_ONLY",
    "RANKER_POLICY: EXACT_KGMON_TWO_GUESSES_NO_POSTHOC_CHANGE",
    "RUNTIME_FORECAST: " + json.dumps(recovery_report["runtime_forecast"], sort_keys=True),
    "EARLY_FORMAT_RELOAD_SELFTEST_GATE: " + ("OK" if early_format_report.get("ok") else "MISSING_OR_FAILED"),
    "CANDIDATE_PICKLE_FIXTURE_GATE: " + ("OK" if candidate_pickle_fixture_report.get("ok") else "MISSING_OR_FAILED"),
    "POST_PRIMARY_EARLY_GATE: " + post_primary_early_gate,
    "POST_PRIMARY_EARLY_GATE_REASONS: " + json.dumps(post_primary_early_reasons, sort_keys=True),
    "DEEP_SKIPPED_BY_EARLY_GATE: " + str(bool(recovery_report.get("deep_skipped_by_early_gate"))),
    "QUALITY_SCANNER_CONSISTENCY_GATE: " + ("OK" if quality_scanner_consistent else "FAILED"),
    "QUALITY_SCANNER_COMPARISON: " + json.dumps(quality_scanner_mismatch, sort_keys=True),
    "PRIMARY_CHECKPOINT_GATE: " + ("OK" if primary_checkpoint_report.get("ok") else "FAILED"),
    "PRIMARY_CHECKPOINT_REPORT: " + json.dumps(primary_checkpoint_report, sort_keys=True),
    "NOTEBOOK_RUNTIME_SECONDS: " + str(notebook_runtime_seconds),
    "HARD_RUNTIME_LIMIT_SECONDS: 34200",
    "PRIMARY_PASS_RUNTIME_SECONDS: " + str(recovery_report["primary_pass"]["elapsed_seconds"]),
    "DEEP_PASS_RUNTIME_SECONDS: " + str(recovery_report["deep_pass"]["elapsed_seconds"] if recovery_report.get("deep_pass") else 0),
    "PRIMARY_STATUS: " + primary_status,
    "DEEP_STATUS: " + deep_status,
    "DEEP_TASK_COUNT: " + str(recovery_report["deep_task_count"]),
    "DEEP_TASKS: " + json.dumps(recovery_report["deep_tasks"], sort_keys=True),
    "PRIMARY_MISSING_CANDIDATE_OUTPUT_COUNT: " + str(len(primary_missing_basekeys)),
    "PRIMARY_PRESENT_CANDIDATE_OUTPUT_RATIO: " + str(round((len(expected_basekeys) - len(primary_missing_basekeys)) / max(len(expected_basekeys), 1), 6)),
    "MINIMUM_PRIMARY_PRESENT_RATIO: 0.94",
    "PRIMARY_MISSING_BASEKEYS: " + json.dumps(sorted(primary_missing_basekeys), sort_keys=True),
    "PRIMARY_STARVATION_COUNT: " + str(len(eligible_basekeys)),
    "PRIMARY_STARVED_BASEKEYS: " + json.dumps(sorted(eligible_basekeys), sort_keys=True),
    "DEEP_SELECTED_BASEKEY_COUNT: " + str(len(used_deep_basekeys)),
    "DEEP_SELECTED_BASEKEYS: " + json.dumps(sorted(used_deep_basekeys), sort_keys=True),
    "FINAL_MISSING_CANDIDATE_OUTPUT_COUNT: " + str(len(final_missing_basekeys)),
    "FINAL_MISSING_BASEKEYS: " + json.dumps(final_missing_basekeys, sort_keys=True),
    "CORRUPT_OUTPUT_FILE_COUNT: " + str(len(corrupt_files)),
    "CORRUPT_OUTPUT_FILES: " + json.dumps(corrupt_files, sort_keys=True),
    "PRIMARY_EXISTING_TOP1_MUTATION_COUNT: 0",
    "WELL_COVERED_PRIMARY_TOP2_MUTATION_COUNT: 0",
    "TOP1_CHANGED_COUNT: " + str(len(top1_changed_keys)),
    "TOP1_CHANGED_KEYS: " + json.dumps(top1_changed_keys, sort_keys=True),
    "SECOND_SLOT_CHANGED_COUNT: " + str(len(second_slot_changed_keys)),
    "SECOND_SLOT_CHANGED_KEYS: " + json.dumps(second_slot_changed_keys, sort_keys=True),
    "COMBINED_CANDIDATE_STARVATION_COUNT: " + str(candidate_starvation_count),
    "TOP1_RANKER_DISAGREEMENT_COUNT: " + str(top1_ranker_disagreement_count),
    "PRIMARY_OUTPUT_FILE_COUNT: " + str(manifest["primary_file_count"]),
    "DEEP_OUTPUT_FILE_COUNT: " + str(manifest["deep_file_count"]),
    "USED_DEEP_OUTPUT_FILE_COUNT: " + str(manifest["used_deep_file_count"]),
    "TASK_OUTPUT_COUNTS_USED: " + json.dumps(dict(sorted(task_counts.items())), sort_keys=True),
    "OUTPUT_MANIFEST_SHA256: " + sha256_file(manifest_path),
    "RANKER_AUDIT_SHA256: " + sha256_file(ranker_audit_path),
    "SUBMISSION_FORMAT_GATE: OK",
    "POST_WRITE_RELOAD_GATE: OK",
    "SUBMISSION_FORMAT_REPORT: " + json.dumps(format_report, sort_keys=True),
    "SELECTED_POLICY: " + selected_policy,
    "SELECTION_SAFETY_FALLBACK_REASONS: " + json.dumps(selection_safety_fallback_reasons, sort_keys=True),
    "HIDDEN_RERUN_RECOVERY_MODE: " + str(competition_rerun_recovery_mode),
    "BASELINE_SUBMISSION_SHA256: " + baseline_sha256,
    "EXPERIMENTAL_SUBMISSION_SHA256: " + experimental_sha256,
    "SUBMISSION_PATH: " + str(submission_path),
    "SUBMISSION_SHA256: " + submission_sha256,
    "REPRODUCIBILITY_SIGNATURE: " + reproducibility_sha256,
    "LOCAL_EVALUATION_MAX_POINTS: " + str(local_evaluation_max_points),
    "BASELINE_LOCAL_EVALUATION_RAW_POINTS: " + str(baseline_local_evaluation_raw_points),
    "BASELINE_LOCAL_EVALUATION_PERCENT: " + str(baseline_local_evaluation_percent),
    "PROGRAM077_PRIMARY_LOCAL_EVALUATION_RAW_POINTS: " + str(program077_local_evaluation_raw_points),
    "PROGRAM077_PRIMARY_LOCAL_EVALUATION_PERCENT: " + str(program077_local_evaluation_percent),
    "LOCAL_EVALUATION_DIAGNOSTIC_NOTE: VISIBLE_EVALUATION_LABELS_ONLY_NOT_KAGGLE_PUBLIC_OR_PRIVATE_AND_NOT_A_SUBMISSION_GATE",
    "DAILY_SUBMISSION_GATE: " + daily_gate,
    "DAILY_SUBMISSION_REASONS: " + json.dumps(daily_reasons, sort_keys=True),
    "DIAGNOSTIC_CONCLUSION: SEPTEMBER_HIDDEN_RERUN_RECOVERY_PRIMARY_ANCHOR",
    "NEXT_RECOMMENDED_ACTION: " + next_action,
    "PRIVATE_SCORE: USER_TO_FILL",
    "＝＝＝＝GPT FB STOP ＝＝＝＝",
]
print("PROGRAM077_SUBMISSION_FORMAT_GATE=OK")
print("PROGRAM077_POST_WRITE_RELOAD_GATE=OK")
print(
    "PROGRAM077_FAIL_CLOSED_SELECTION_GATE="
    + ("OK" if not selection_safety_fallback_reasons else "FAILED_BASELINE_FORCED")
)
print("PROGRAM077_SELECTION_AUDIT=" + json.dumps({
    "eligible": len(eligible_basekeys),
    "used_deep": len(used_deep_basekeys),
    "top1_changed": len(top1_changed_keys),
    "second_changed": len(second_slot_changed_keys),
    "selected_policy": selected_policy,
    "selection_safety_fallback_reasons": selection_safety_fallback_reasons,
    "corrupt_files": len(corrupt_files),
}, sort_keys=True))
print("\n".join(gpt_fb_lines))
'''

finalize_path.write_text(bootstrap_source + finalize_body, encoding="utf-8")
finalize_env = os.environ.copy()
finalize_env.update({
    "PROGRAM077_LEGACY_ROOT": str(legacy_root),
    "PROGRAM077_OVERLAY_ROOT": str(overlay_root),
    "PYTHONPATH": run_plan["runtime_pythonpath"],
    "UNSLOTH_DISABLE_STATISTICS": "1",
    "TRITON_PTXAS_PATH": run_plan["ptxas_path"],
    "PYTHONHASHSEED": run_plan["seed_contract"]["python_hash_seed"],
    "PROGRAM077_BASE_SEED": str(run_plan["seed_contract"]["base_seed"]),
    "PROGRAM077_SEED_POLICY": "rng_only_no_backend_change",
    "HF_HUB_OFFLINE": "1",
    "TRANSFORMERS_OFFLINE": "1",
})
def write_program077_emergency_submission(challenge_path, submission_path, cause):
    import hashlib
    import json
    import os
    from pathlib import Path

    challenge_path = Path(challenge_path)
    submission_path = Path(submission_path)
    challenge = json.loads(challenge_path.read_text(encoding="utf-8"))
    if not isinstance(challenge, dict) or not challenge:
        raise ValueError("PROGRAM077_EMERGENCY_CHALLENGE_INVALID")
    submission = {}
    checked_tests = 0
    for task_id, task in sorted(challenge.items()):
        tests = task.get("test", [])
        if not isinstance(tests, list):
            raise ValueError(f"PROGRAM077_EMERGENCY_TESTS_INVALID={task_id}")
        submission[task_id] = [
            {"attempt_1": [[0]], "attempt_2": [[0]]}
            for _ in tests
        ]
        checked_tests += len(tests)
    temporary_path = submission_path.with_name(submission_path.name + ".tmp")
    temporary_path.write_text(
        json.dumps(submission, sort_keys=True, separators=(",", ":"), allow_nan=False),
        encoding="utf-8",
    )
    os.replace(temporary_path, submission_path)
    reloaded = json.loads(submission_path.read_text(encoding="utf-8"))
    if reloaded != submission or set(reloaded) != set(challenge):
        raise RuntimeError("PROGRAM077_EMERGENCY_RELOAD_FAILED")
    digest = hashlib.sha256(submission_path.read_bytes()).hexdigest()
    report = {
        "ok": True,
        "cause": str(cause)[:2000],
        "challenge_task_count": len(challenge),
        "checked_test_count": checked_tests,
        "checked_attempt_count": checked_tests * 2,
        "submission_path": str(submission_path),
        "submission_sha256": digest,
        "policy": "ALL_ZERO_SCHEMA_VALID_LAST_RESORT",
    }
    (submission_path.parent / "program077_emergency_recovery_report.json").write_text(
        json.dumps(report, sort_keys=True, separators=(",", ":")),
        encoding="utf-8",
    )
    return report


finalize = None
finalize_failure = None
try:
    finalize = subprocess.run(
        [str(python_executable), str(finalize_path)],
        env=finalize_env,
        cwd="/kaggle/working",
        text=True,
        capture_output=True,
        timeout=900,
    )
except subprocess.TimeoutExpired as exc:
    finalize_failure = f"TIMEOUT_AFTER_{exc.timeout}_SECONDS"
    if exc.stdout:
        print(exc.stdout, end="")
    if exc.stderr:
        print(exc.stderr, file=sys.stderr, end="")

if finalize is not None:
    if finalize.stdout:
        print(finalize.stdout, end="")
    if finalize.returncode != 0:
        if finalize.stderr:
            print(finalize.stderr, file=sys.stderr, end="")
        finalize_failure = f"EXIT_CODE_{finalize.returncode}"

if finalize_failure is not None:
    emergency_report = write_program077_emergency_submission(
        run_plan["challenge_path"],
        "/kaggle/working/submission.json",
        finalize_failure,
    )
    print("PROGRAM077_EMERGENCY_SUBMISSION_WRITTEN=" + json.dumps(emergency_report, sort_keys=True))
